## Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import time
import ipyleaflet
import warnings
import ipywidgets as widgets
from IPython.display import display
from sklearn.inspection import permutation_importance
import tempfile
import ee
import geemap
region_json = None
from pprint import pprint

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, cohen_kappa_score

from sklearn.ensemble import RandomForestClassifier
try:
    ee.Initialize(project='boreal-doodad-496509-b4')
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project='boreal-doodad-496509-b4')

Map = geemap.Map(basemap='Esri.WorldStreetMap', attribution_control=False)
pd.options.display.float_format = '{:,.2f}'.format

## Widgets components

In [2]:

label_layout = widgets.Layout(display="flex", justify_content="flex-end", width="90%")

box_layout = widgets.Layout(display='flex',
                flex_flow='column',
                align_items='center')

sidebar_layout = widgets.Layout(width='98%')

desc_width = {'description_width': '120px'}

### Satellite and bands

In [3]:
satellite_list = [
    'Landsat 7 SR',

    'Landsat 8 SR',
    'Landsat 9 SR',

    'Landsat 7 TOA',

    'Landsat 8 TOA',
    'Landsat 9 TOA',
    'Sentinel-2 SR',
    'Sentinel-2 TOA']

collections_list = [
    'LANDSAT/LE07/C02/T1_L2',

    'LANDSAT/LC08/C02/T1_L2',
    'LANDSAT/LC09/C02/T1_L2',

    'LANDSAT/LE07/C02/T1_TOA',
    'LANDSAT/LC08/C02/T1_TOA',
    'LANDSAT/LC09/C02/T1_TOA',
    'COPERNICUS/S2_SR_HARMONIZED',
    'COPERNICUS/S2_HARMONIZED']

bands_list = [
    ['SR_B'+str(i) for i in range(1,6)] + ['SR_B7', 'ST_B6'],
    ['SR_B'+str(i) for i in range(1,8)] + ['ST_B10'],
    ['SR_B'+str(i) for i in range(1,8)] + ['ST_B10'],

    ['B'+str(i) for i in range(1,6)] + ['B7', 'B8'],
    ['B'+str(i) for i in range(1,12)],
    ['B'+str(i) for i in range(1,12)],
    ['B'+str(i) for i in range(1,10)] + ['B8A', 'B11', 'B12'],
    ['B'+str(i) for i in range(1,13)] + ['B8A'],
]

w_satellite = widgets.Dropdown(
    options=satellite_list,
    value='Landsat 9 SR',
    description='Satellite',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

ind = satellite_list.index(w_satellite.value)

w_select_bands = widgets.SelectMultiple(
    options=bands_list[ind],
    value=bands_list[ind][1:6],
    rows=6,
    description='Bands',
    style=desc_width,
    layout=sidebar_layout,
    disabled=False
)

def satellite_handler(change):
  ind = satellite_list.index(change.new)
  w_select_bands.options = bands_list[ind]
  w_select_bands.value = bands_list[ind][1:6]

w_satellite.observe(satellite_handler, names='value')



### Region of interest

Có nhiều cách khác nhau để bạn có thể tạo một vùng chọn (region) nhằm phục vụ việc tạo lập tập mẫu huấn luyện:

Vẽ một hình khối (ví dụ: hình chữ nhật) trực tiếp trên bản đồ và sau đó sử dụng lệnh: region = Map.user_roi

Xác định một đối tượng hình học (geometry) cụ thể, ví dụ: region = ee.Geometry.Rectangle([-122.6003, 37.4831, -121.8036, 37.8288])

Tạo một vùng đệm (buffer zone) quanh một điểm, ví dụ: region = ee.Geometry.Point([-122.4439, 37.7538]).buffer(10000)

Nếu bạn không xác định vùng chọn, hệ thống sẽ mặc định sử dụng toàn bộ phạm vi của hình ảnh (image footprint).
```
image = (
  ee.ImageCollection(collection)
    .filterBounds(region)
    .filterDate(start_date, end_date)
    .map(maskL8sr)
    .median()
    .select(bands)
    .clip(region)
)
```

In [4]:

style = {'description_width': 'initial'}
layout = widgets.Layout(width='98%')

def _handle_file_upload(uploaded_files, default_name='uploaded_file'):
    """Save uploaded file to a system temp folder and return absolute path."""
    if not uploaded_files:
        return None
    try:
        if isinstance(uploaded_files, tuple):
            file_info = uploaded_files[0]
            fname = file_info.get('name', default_name)
            content = file_info.get('content')
        elif isinstance(uploaded_files, dict):
            key = list(uploaded_files.keys())[0]
            file_info = uploaded_files[key]
            fname = key
            content = file_info.get('content')
        else:
            file_info = uploaded_files[0]
            fname = file_info.get('name', default_name)
            content = file_info.get('content')

        fname = os.path.basename(fname) or default_name
        
    
        save_dir = tempfile.gettempdir()
        save_path = os.path.join(save_dir, fname)

        with open(save_path, 'wb') as f:
            f.write(content)

        return os.path.abspath(save_path)

    except Exception as e:
        print(f'Lỗi khi lưu file: {e}')
        return None

region_options = [
    'Select an option', 
    'Draw shapes on map', 
    'Input point and buffer', 
    'Rectangle from BBox', 
    'Upload GeoJSON', 
]

w_region = widgets.Dropdown(
    options=region_options,
    value='Select an option',
    description='Region',
    style=style,
    layout=layout
)

w_region_detail = widgets.VBox()

def regions_handler(change):
    global region_json 
    
    if change.new == 'Draw shapes on map':
        w_region_detail.children = [widgets.Label(value="Hãy vẽ lên map.")]

    elif change.new == 'Input point and buffer':
        w_point = widgets.Text(description='Lon, Lat', style=style, layout=layout)
        w_buffer = widgets.FloatText(description='Buffer (km)', style=style, layout=layout)
        w_region_detail.children = [w_point, w_buffer]

    elif change.new == 'Rectangle from BBox':
        w_points = widgets.Text(description='BBox (W, S, E, N)', style=style, layout=layout)
        w_region_detail.children = [w_points]

    elif change.new == 'Upload GeoJSON':
        w_uploader = widgets.FileUpload(
            accept='.geojson', 
            multiple=False,
            description='Upload GeoJSON'
        )
        
        w_upload_output = widgets.Output()

        def on_upload_change(change):
            global region_json
            w_upload_output.clear_output()
            
            with w_upload_output:
                result = _handle_file_upload(w_uploader.value, 'uploaded_region.geojson')
                if result:
                    region_json = result
                    print(f"Đã upload thành công: {region_json}")

        w_uploader.observe(on_upload_change, names='value')
        w_region_detail.children = [w_uploader, w_upload_output]

    else:
        w_region_detail.children = [widgets.Label(value="Default to current map boundary.")]

w_region.observe(regions_handler, names='value')
w_regions = widgets.VBox([w_region, w_region_detail])

display(w_regions)

### DatePicker & temporal interval

In [5]:
w_start_date = widgets.DatePicker(
    description='Start date',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

w_end_date = widgets.DatePicker(
    description='End date',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)



temporal_options = [
    'All dates', 
    'Yearly', 
    'Monthly', 
    'Daily', 
]

w_temporal = widgets.Dropdown(
    options=temporal_options,
    value='All dates',
    description='Composite interval',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

w_temporal_detail = widgets.VBox()

def temporal_handler(change):
  if change.new == 'All dates':
    w_temporal_detail.children = [widgets.Label(value="Median composite for the entire interval",
                                                layout=label_layout)]
  elif change.new == 'Yearly':
    w_temporal_detail.children = [widgets.Label(value="Yearly median composite",
                                                layout=label_layout)]
  elif change.new == 'Monthly':
    w_temporal_detail.children = [widgets.Label(value="Monthly median composite",
                                                layout=label_layout)]
  elif change.new == 'Daily':
    w_temporal_detail.children = [widgets.Label(value="Daily mosaic",
                                                layout=label_layout)]

w_temporal.observe(temporal_handler, names='value')

w_dates = widgets.VBox([
    w_start_date,
    w_end_date,
    w_temporal,
    w_temporal_detail
])


### Spectal indices & other variables

In [6]:
w_spectral_indices = widgets.SelectMultiple(
    options=['None', 'EVI', 'MNDWI', 'NBR', 'NDBI', 'BSI'],
    value=['None'],
    description='Spectral indices',
    rows=6,
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

w_variables = widgets.SelectMultiple(
    options=['Elevation', 'Slope', 'Aspect'],
    value=[],
    description='Other variables',
    rows=3,
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

w_split_ratio = widgets.BoundedFloatText(
    value=0.7,
    min=0.1,
    max=1,
    step=0.01,
    description='Split ratio',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

In [7]:
# === Advanced Module Widgets ===

# CloudScore+ threshold for S2_HARMONIZED cloud masking
w_cloudscore_threshold = widgets.BoundedFloatText(
    value=0.6,
    min=0.0,
    max=1.0,
    step=0.05,
    description='CloudScore+ threshold',
    style=desc_width,
    layout=sidebar_layout
)

# SAR Fusion widgets
w_sar_fusion = widgets.Checkbox(
    description='SAR fusion (Sentinel-1)',
    value=True,
    style=desc_width,
    layout=sidebar_layout
)

w_speckle_radius = widgets.BoundedIntText(
    value=50,
    min=10,
    max=200,
    step=10,
    description='Speckle filter radius (m)',
    style=desc_width,
    layout=sidebar_layout
)

# Harmonic Regression widgets
w_harmonic_regression = widgets.Checkbox(
    description='Harmonic regression',
    value=False,
    style=desc_width,
    layout=sidebar_layout
)

w_harmonic_order = widgets.Dropdown(
    options=[1, 2],
    value=2,
    description='Harmonic order',
    style=desc_width,
    disabled=True,
    layout=sidebar_layout
)

# Phenology Metrics widget
w_phenology_metrics = widgets.Checkbox(
    description='Phenology metrics',
    value=False,
    style=desc_width,
    disabled=True,
    layout=sidebar_layout
)

w_phenology_warning = widgets.HTML(value='')
w_harmonic_warning = widgets.HTML(value='')

def harmonic_toggle_handler(change):
    if change.new:  # Khi bật Harmonic regression
        if 'EVI' not in list(w_spectral_indices.value):
            w_harmonic_regression.value = False
            w_harmonic_warning.value = (
                '<span style="color:red;">'
                '⚠️ Harmonic regression yêu cầu chỉ số EVI. '
                'Vui lòng chọn EVI trong danh sách Spectral indices trước.'
                '</span>'
            )
        else:
            w_harmonic_warning.value = ''
            w_harmonic_order.disabled = False
            w_phenology_metrics.disabled = False
    else:
        w_harmonic_warning.value = ''
        w_harmonic_order.disabled = True
        w_phenology_metrics.value = False
        w_phenology_metrics.disabled = True
        w_phenology_warning.value = ''

def phenology_toggle_handler(change):
    if change.new:  # Khi bật Phenology metrics
        if 'EVI' not in list(w_spectral_indices.value):
            w_phenology_metrics.value = False
            w_phenology_warning.value = (
        '<span style="color:red;">'
        '⚠️ Phenology yêu cầu chỉ số EVI. '
        'Vui lòng chọn EVI trong danh sách Spectral indices trước.'
        '</span>'
    )
        else:
            w_phenology_warning.value = ''
    else:
        w_phenology_warning.value = ''

w_harmonic_regression.observe(harmonic_toggle_handler, names='value')
w_phenology_metrics.observe(phenology_toggle_handler, names='value')

# Spatial Block CV widgets
w_spatial_block_cv = widgets.Checkbox(
    description='Spatial block CV',
    value=False,
    style=desc_width,
    layout=sidebar_layout
)

w_block_size_km = widgets.BoundedIntText(
    value=10,
    min=1,
    max=100,
    step=1,
    description='Block size (km)',
    style=desc_width,
    layout=sidebar_layout
)

w_n_folds = widgets.BoundedIntText(
    value=5,
    min=2,
    max=20,
    step=1,
    description='Number of folds',
    style=desc_width,
    layout=sidebar_layout
)

w_block_warning = widgets.HTML(value='')

def block_size_handler(change):
    if change.new < 5:
        w_block_warning.value = '<span style="color:orange">Warning: Block size < 5 km may not eliminate spatial autocorrelation</span>'
    else:
        w_block_warning.value = ''

w_block_size_km.observe(block_size_handler, names='value')
w_spatial_cv_detail = widgets.VBox([
    w_block_size_km,
    w_n_folds,
    w_block_warning
])

w_split_ratio_box = widgets.VBox([w_split_ratio])

def sync_validation_widgets(change=None):
    if w_spatial_block_cv.value:
        # Bật Spatial Block CV -> hiện block settings, ẩn split ratio
        w_spatial_cv_detail.layout.display = "block"
        w_split_ratio_box.layout.display = "none"
        w_split_ratio.disabled = True
        w_block_size_km.disabled = False
        w_n_folds.disabled = False
    else:
        # Tắt Spatial Block CV -> hiện split ratio, ẩn block settings
        w_spatial_cv_detail.layout.display = "none"
        w_split_ratio_box.layout.display = "block"
        w_split_ratio.disabled = False
        w_block_size_km.disabled = True
        w_n_folds.disabled = True

w_spatial_block_cv.observe(sync_validation_widgets, names='value')
sync_validation_widgets()
# Change Detection widgets
w_change_detection = widgets.Checkbox(
    description='Change detection',
    value=False,
    style=desc_width,
    layout=sidebar_layout
)

w_period1_start = widgets.DatePicker(
    description='Period 1 start',
    style=desc_width,
    layout=sidebar_layout
)

w_period1_end = widgets.DatePicker(
    description='Period 1 end',
    style=desc_width,
    layout=sidebar_layout
)

w_period2_start = widgets.DatePicker(
    description='Period 2 start',
    style=desc_width,
    layout=sidebar_layout
)

w_period2_end = widgets.DatePicker(
    description='Period 2 end',
    style=desc_width,
    layout=sidebar_layout
)

w_min_transition_area = widgets.BoundedFloatText(
    value=1.0,
    min=0.0,
    max=100.0,
    step=0.1,
    description='Min transition area (km²)',
    style=desc_width,
    layout=sidebar_layout
)

w_change_detection_detail = widgets.VBox()

def change_detection_handler(change):
    if change.new:
        w_change_detection_detail.children = [
            w_period1_start, w_period1_end,
            w_period2_start, w_period2_end,
            w_min_transition_area
        ]
    else:
        w_change_detection_detail.children = []

w_change_detection.observe(change_detection_handler, names='value')


### Multi-source checkbox

If multi-source classification, select multiple satellites and their bands.
```
pprint(multiSource)

{'Landsat 9 SR': ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6'],
 'Sentinel-2 TOA': ['B2', 'B3', 'B4', 'B5', 'B6']}
```

In [8]:
w_multiSource = widgets.Checkbox(description='Multi-source',
                                 value=False,
                                 style=desc_width,
                                 layout=sidebar_layout)

multiSource = {}

def multiSourceChanged(value):
    if value['new'] == True:
      w_btn_addSource.layout.display = "block"
      with w_btn_addSource_output:
        print('Select satellite/bands, then click "Add source" button.')
    else:
      w_btn_addSource_output.clear_output()
      multiSource.clear()
      #w_btn_addSource.layout.visibility = "hidden"
      w_btn_addSource.layout.display = "none"

w_multiSource.observe(multiSourceChanged, names='value')

w_btn_addSource = widgets.Button(description='Add source',
                            button_style='danger',
                            tooltip="Add satellite/bands",
                            #layout=widgets.Layout(visibility='hidden')
                            layout=widgets.Layout(display='none')
                            )

w_btn_addSource_output = widgets.Output()
def btn_addSource_handler(obj):
    w_btn_addSource_output.clear_output()
    with w_btn_addSource_output:
      #print('Added satellite/bands for', w_satellite.value)
      multiSource[w_satellite.value] = list(w_select_bands.value)
      print('Current selection')
      pprint(multiSource)

w_btn_addSource.on_click(btn_addSource_handler)

w_source = widgets.VBox([#widgets.HBox([w_multiSource, w_btn_addSource]),
                         w_multiSource,
                         w_btn_addSource,
                         w_btn_addSource_output])



### Classifier


In [9]:
classifier_list = [
    'RandomForest',
]

w_rf_param = [
    widgets.BoundedIntText(
        value=100,
        min=1,
        max=1000,
        step=1,
        description='numberOfTrees',
        disabled=True,
        style=desc_width,
        layout=sidebar_layout
    ),
    widgets.BoundedIntText(
        value=10,
        min=1,
        max=1000,
        step=1,
        description='variablesPerSplit',
        disabled=True,
        style=desc_width,
        layout=sidebar_layout
    ),
    widgets.BoundedIntText(
        value=5,
        min=1,
        max=100,
        step=1,
        description='minLeafPopulation',
        disabled=True,
        style=desc_width,
        layout=sidebar_layout
    ),
    widgets.BoundedFloatText(
        value=0.5,
        min=0.01,
        max=1,
        step=0.01,
        description='bagFraction',
        disabled=True,
        style=desc_width,
        layout=sidebar_layout
    ),
    widgets.BoundedIntText(
        value=1000,
        min=1,
        max=1000,
        step=1,
        description='maxNodes',
        disabled=True,
        style=desc_width,
        layout=sidebar_layout
    ),
]

classifier_param ={
    'RandomForest': w_rf_param
}

w_classifier_select = widgets.Dropdown(
    options=classifier_list,
    value='RandomForest',
    description='Select classifier',
    style=desc_width,
    layout=sidebar_layout
)

w_classifier_customize = widgets.Checkbox(description='Customize parameters',
                                          value=False,
                                          style=desc_width,
                                          layout=sidebar_layout)

w_classifier_param = widgets.VBox()


def classifier_handler(change):
  if w_classifier_customize.value:
    w_classifier_param.children = classifier_param[change.new]
    for w in w_classifier_param.children:
      w.disabled=False

w_classifier_select.observe(classifier_handler, names='value')

def classifier_customizeChanged(value):
    if value['new'] == False:
      w_classifier_param.children=[]
    else:
      w_classifier_param.children = classifier_param[w_classifier_select.value]
      for w in w_classifier_param.children:
        w.disabled=False

w_classifier_customize.observe(classifier_customizeChanged, names='value')

w_classifier = widgets.VBox([w_classifier_select, w_classifier_customize, w_classifier_param],
                            layout={'border': '1px solid green'})



### Widget output control on map

In [10]:
w_map_output = widgets.Output(layout={'border': '1px solid black',
                                      #'width': '200px'
                                      })
output_control = ipyleaflet.WidgetControl(widget=w_map_output, position='bottomleft')
Map.add_control(output_control)

### Workflow buttons

In [11]:


w_classified_flag = widgets.Checkbox(description='Image classified?', value=False)

w_classified_tmpflag = widgets.Checkbox(description='Temporal composite classified?', value=False)


# Temporal-analysis cache/state
colMosaic_tmp_classified = None
listOfImages = None
col_dates_list = None
df_tmpZonal = None


w_btn_workflow_log = widgets.Output(layout={'border': '2px solid blue',
                                            'height': '200px',
                                            'overflow': 'auto'})

w_btn_classify = widgets.Button(description='Classify',
                           button_style='danger',
                           tooltip="Filter image collection and classify image",
                           layout=widgets.Layout(width='80%'))

def btn_classify_handler(obj):
    global imageMedian

    w_btn_workflow_log.clear_output()
    w_map_output.clear_output()

    global colMosaic_tmp_classified, listOfImages, col_dates_list, df_tmpZonal

    w_classified_flag.value = False
    w_classified_tmpflag.value = False
    colMosaic_tmp_classified = None
    listOfImages = None
    col_dates_list = None
    df_tmpZonal = None

    clear_analysis_cache()

    for g in ['transition_df', 'change_map', 'classified_period1', 'classified_period2']:
        if g in globals():
            del globals()[g]

    with w_btn_workflow_log:
        print('Step 1/3: Get composite image')
        imageMedian = getImage()
        if imageMedian is None:
            print('\nSTOPPED: getImage() returned None.')
            return

        print('\nStep 2/3: Get samples data')
        training_local = getSamplesData(imageMedian)
        if training_local is None:
            print('\nSTOPPED: getSamplesData() returned None.')
            return

        print('\nStep 3/3: Classify the image')
        classified_local = runGeeClassifier(imageMedian, training_local)
        if classified_local is None:
            print('\nSTOPPED: runGeeClassifier() returned None.')
            return

        print('\nDONE.')

    w_classified_flag.value = True
    w_map_output.clear_output()
    with w_btn_workflow_log:
        if w_temporal.value != 'All dates':
            try:
                getClassifiedCollections()  # populate col_dates_list sớm
            except Exception as e:
                print(f'Could not pre-load dates: {e}')
w_btn_classify.on_click(btn_classify_handler)


   

### Samples from labeled points uploaded
Tải lên tệp CSV, mỗi hàng đại diện cho một mẫu (lon, lat, label, date (nếu có)).


In [12]:


sample_csv = None
CLASS_NAMES = None
CLASS_PALETTE = None



w_upload_sample = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload samples',
    style=desc_width
)

w_includeLabels = widgets.Checkbox(
    description='Class names included',
    value=False,
    style=desc_width
)

w_class_labels = widgets.Text(
    description='Class names',
    placeholder='Comma-separated names',
    layout=widgets.Layout(width='98%'),
    style=desc_width
)

w_randomColors = widgets.Checkbox(
    value=True,
    description='Use random colors',
    style=desc_width,
    layout=sidebar_layout
)

w_palette = widgets.VBox(layout=widgets.Layout(display='none'))

w_btn_sample_output = widgets.Output()

w_sample1 = widgets.VBox([
    w_upload_sample,
    w_btn_sample_output,
    w_includeLabels,
    w_class_labels,
    w_randomColors,
    w_palette
])



def init_class_palette():
    """Create/update per-class ColorPickers using classes read from the uploaded CSV."""
    global CLASS_NAMES, CLASS_PALETTE

    res = getClassLabelsFromFile()

    if isinstance(res, tuple) and len(res) == 2:
        CLASS_NAMES = res[0]

    if CLASS_NAMES is None:
        return

    if (CLASS_PALETTE is None) or (len(CLASS_PALETTE) != len(CLASS_NAMES)):
        CLASS_PALETTE = [
            "#" + "".join([random.choice("0123456789ABCDEF") for _ in range(6)])
            for _ in range(len(CLASS_NAMES))
        ]

    # Build pickers
    w_palette.children = [
        widgets.ColorPicker(
            description=name,
            value=color,
            disabled=False,
            style=desc_width,
            layout=sidebar_layout
        )
        for name, color in zip(CLASS_NAMES, CLASS_PALETTE)
    ]


def randomColorsChanged(change):
    global CLASS_NAMES, CLASS_PALETTE

    res = getClassLabelsFromFile()
    if isinstance(res, tuple) and len(res) == 2:
        CLASS_NAMES = res[0]

    if CLASS_NAMES is None:
        return

    if change.get("new") is False:
        init_class_palette()
        w_palette.layout.display = "block"
        CLASS_PALETTE = [c.value for c in w_palette.children] if w_palette.children else CLASS_PALETTE
    else:
        w_palette.layout.display = "none"
w_randomColors.observe(randomColorsChanged, names='value')




def on_sample_upload_change(change):
    global sample_csv

    w_btn_sample_output.clear_output()

    with w_btn_sample_output:
        result = _handle_file_upload(
            w_upload_sample.value,
            'uploaded_samples.csv'
        )

        if result:
            sample_csv = result
            print(f"Đã upload thành công: {sample_csv}")
            print("Sample format: (lon, lat, label, date)")

            init_class_palette()

            if w_randomColors.value is False:
                w_palette.layout.display = "block"

w_upload_sample.observe(on_sample_upload_change, names='value')



def includeLabelsChanged(change):
    if change['new'] is True:
        w_class_labels.layout.display = "none"
    else:
        w_class_labels.layout.display = "block"

w_includeLabels.observe(includeLabelsChanged, names='value')

### Samples from LULC products

In [13]:
w_sample_select = widgets.Dropdown(
    options=['Upload samples', 'Sample LULC products'],
    value='Upload samples',
    description='Labeled data',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

def sample_select_handler(change):
  if change.new == 'Sample LULC products':
    w_sample1.layout.display= "none"
    w_sample2.layout.display= "block"
  else:
    w_sample2.layout.display= "none"
    w_sample1.layout.display= "block"
  # Clear output widgets
  w_btn_workflow_log.clear_output()
  w_map_output.clear_output()

w_sample_select.observe(sample_select_handler, names='value')

lulc_list = [
    'Dynamic World V1',
    'ESA WorldCover 10m v100/200',
    'ESRI Global Land Cover',
]

w_lulc_select = widgets.Dropdown(
    options=lulc_list,
    value='Dynamic World V1',
    description='LULC map',
    disabled=False,
    style=desc_width,
    layout=sidebar_layout
)

w_numPoints = widgets.BoundedIntText(
  value=1000,
  min=100,
  max=100000,
  step=100,
  description='Labels total',
  style=desc_width,
  layout=sidebar_layout
)

w_stratifiedSample = widgets.Checkbox(value=False,
                                      description='Stratified sample',
                                      style=desc_width,
                                      layout=sidebar_layout)

def stratifiedSampleChanged(value):
    if value['new'] == True:
      w_numPoints.description = 'Labels per class'
    else:
      w_numPoints.description = 'Labels total'

w_stratifiedSample.observe(stratifiedSampleChanged, names='value')

w_sample2 = widgets.VBox([w_lulc_select, w_stratifiedSample, w_numPoints],
                         layout=widgets.Layout(display='none'))

w_sample = widgets.VBox([w_sample_select,
                         w_sample1,
                         w_sample2])


In [14]:
def getClassLabelsFromFile():
    """Read uploaded sample CSV, normalize labels, and return class names + EE FeatureCollection."""
    global LABEL, CLASS_NAMES, LABEL_DATA

    if sample_csv is None:
        print('Upload samples first.')
        return None, None

    if (w_includeLabels.value is False) and (not w_class_labels.value.strip()):
        print('Incomplete input. If sample file does not include class names, enter them as a comma-separated list.')
        return None, None

    try:
        df = pd.read_csv(sample_csv)
    except Exception as e:
        print(f'Cannot read sample CSV: {e}')
        return None, None

    if df.shape[1] < 3:
        print('CSV must have at least 3 columns: longitude, latitude, label')
        return None, None

    # Chuẩn hóa tên cột thật sự, tránh lỗi khoảng trắng đầu/cuối
    df = df.rename(columns=lambda c: str(c).strip())

    col_lookup = {str(c).lower(): c for c in df.columns}

    lon_candidates = ['lon', 'lng', 'longitude', 'x']
    lat_candidates = ['lat', 'latitude', 'y']
    date_candidates = ['date']

    lon_col = next((col_lookup[c] for c in lon_candidates if c in col_lookup), None)
    lat_col = next((col_lookup[c] for c in lat_candidates if c in col_lookup), None)
    date_col = next((col_lookup[c] for c in date_candidates if c in col_lookup), None)

    if lon_col is None or lat_col is None:
        print('Warning: Could not detect longitude/latitude columns by name. Assuming column order: longitude, latitude, label.')
        lon_col = df.columns[0]
        lat_col = df.columns[1]

    reserved = {str(lon_col).lower(), str(lat_col).lower()}
    if date_col is not None:
        reserved.add(str(date_col).lower())

    label_candidates = [c for c in df.columns if str(c).lower() not in reserved]
    if not label_candidates:
        print('Error: Could not detect a label column (all columns are lon/lat/date).')
        return None, None

    LABEL = label_candidates[0]

    if date_col is not None:
        try:
            dt = pd.to_datetime(
                df[date_col].astype(str).str.strip(),
                format='%d/%m/%Y',
                errors='coerce'
            )
            invalid_dates = int(dt.isna().sum())
            if invalid_dates > 0:
                print(f'Warning: {invalid_dates} rows have invalid date values and were kept as empty dates.')

            df[date_col] = dt.dt.strftime('%Y-%m-%d')
            df['month'] = dt.dt.strftime('%Y-%m')
            df['year'] = dt.dt.year

            # Giữ rỗng cho các dòng date lỗi
            df.loc[dt.isna(), 'month'] = None
            df.loc[dt.isna(), 'year'] = None
        except Exception as e:
            print(f'Warning: Could not normalize date column: {e}')

    raw_labels = df[LABEL]

    if w_includeLabels.value:
        class_names = sorted(pd.Series(raw_labels).dropna().astype(str).str.strip().unique().tolist())
        if not class_names:
            print('No class names found in the label column.')
            return None, None

        label_map = {name: idx for idx, name in enumerate(class_names)}
        df[LABEL] = pd.Series(raw_labels).astype(str).str.strip().map(label_map)

        if df[LABEL].isna().any():
            print('Error: Some labels could not be mapped to class ids.')
            return None, None

        CLASS_NAMES = class_names

    else:
        class_names = [s.strip() for s in w_class_labels.value.split(',') if s.strip()]
        if not class_names:
            print('Class names list is empty.')
            return None, None

        numeric_labels = pd.to_numeric(df[LABEL], errors='coerce')
        if numeric_labels.isna().any():
            print('Error: Label column must contain numeric class ids when "Class names included" is unchecked.')
            return None, None

        numeric_labels = numeric_labels.astype(int)
        max_label = int(numeric_labels.max()) if len(numeric_labels) else -1
        min_label = int(numeric_labels.min()) if len(numeric_labels) else 0

        if min_label < 0:
            print('Error: Label ids must be >= 0.')
            return None, None

        if max_label >= len(class_names):
            print(f'Error: Found label id {max_label}, but only {len(class_names)} class names were provided.')
            return None, None

        df[LABEL] = numeric_labels
        CLASS_NAMES = class_names

    try:
        LABEL_DATA = geemap.df_to_ee(df, latitude=lat_col, longitude=lon_col)
    except Exception as e:
        print(f'Error converting CSV to EE FeatureCollection: {e}')
        return None, None

    return CLASS_NAMES, LABEL_DATA


In [15]:
def getClassLabelsFromLC():
  """ Sample exisiting LULC products for labels """

  global LABEL, CLASS_NAMES, LABEL_DATA, CLASS_PALETTE

  from geemap import legends

  if w_lulc_select.value == 'Dynamic World V1':
    legend = geemap.legends.builtin_legends['Dynamic_World']
  elif w_lulc_select.value == 'ESA WorldCover 10m v100/200':
    legend = geemap.legends.builtin_legends['ESA_WorldCover']
  elif w_lulc_select.value == 'ESRI Global Land Cover':
    legend = geemap.legends.builtin_legends['ESRI_LandCover_TS']

  LABEL = 'Land cover' # hard-coded
  CLASS_NAMES = [name.split(' ', 1)[1] for name in legend.keys()]
  CLASS_LABELS = [int(name.split(' ', 1)[0]) for name in legend.keys()]
  CLASS_PALETTE = list(legend.values())
  CLASS_PALETTE = [c if c.startswith('#') else '#'+c for c in CLASS_PALETTE]

  if w_lulc_select.value == 'Dynamic World V1':
    lc = geemap.dynamic_world(region, start_date, end_date, True, return_type='class')

    lc = lc.rename(LABEL)

  elif w_lulc_select.value == 'ESA WorldCover 10m v100/200':
    if start_date.split('-')[0] == '2020':
      lc = ee.Image('ESA/WorldCover/v100/2020')
    elif start_date.split('-')[0] == '2021':
      lc = ee.Image('ESA/WorldCover/v200/2021')
    else:
      print('ESA WorldCover 10m: v100 is for 2020, v200 is for 2021. Year 2021 used instead.')
      lc = ee.Image('ESA/WorldCover/v200/2021')

    remapped_labels = list(range(0, len(CLASS_LABELS)))
    lc = lc.clip(region).remap(CLASS_LABELS, remapped_labels).rename(LABEL)

  elif w_lulc_select.value == 'ESRI Global Land Cover':
    lc = ee.ImageCollection("projects/sat-io/open-datasets/landcover/ESRI_Global-LULC_10m_TS")

    year = start_date.split('-')[0]
    if int(year) >= 2017 and int(year) <= 2024:
      pass
    else:
      year = '2024'
      print('ESRI Global Land Cover valid years 2017-2024. Year 2024 used instead.')
    lc = lc.filterDate(year+'-01-01', year+'-12-31').mosaic().clip(region)

    # Remap labels and rename band
    remapped_labels = list(range(0, len(CLASS_LABELS)))
    lc = lc.remap(CLASS_LABELS, remapped_labels).rename(LABEL)

  if w_stratifiedSample.value:
    print('Stratified sampling', w_lulc_select.value, '...')
    LABEL_DATA = lc.stratifiedSample(
        **{
            'numPoints': w_numPoints.value,
            'classBand': LABEL,
            'region': region,
            'scale': 10,
            'seed': seed,
            'geometries': True,  # Set this to False to ignore geometries
        }
    )
  else:
    print('Random sampling', w_lulc_select.value, '...')
    LABEL_DATA = lc.sample(
        **{
            'region': region,
            'numPixels': w_numPoints.value,
            'scale': 10,
            'seed': seed,
            'geometries': True,
            'tileScale': 16
        }
    )

  vis_param = {'min': 0, 'max': len(CLASS_PALETTE)-1, 'palette': CLASS_PALETTE}
  Map.addLayer(lc, vis_param, w_lulc_select.value)

  w_map_output.clear_output()
  with w_map_output:
    text = 'Added LULC layer to sample from.'
    htmlWidget = widgets.HTML(value = f"<b><font color='red'>{text}</b>")
    display(htmlWidget)

  return LABEL_DATA, CLASS_PALETTE


## Workflow functions

### Get ROI

In [16]:



def getRegion():
    global region_json # Bắt buộc phải khai báo dòng này để dùng biến toàn cục
    
    region = None # Mặc định
    
    if w_region.value == 'Draw shapes on map':
        print('Use geometry drawn on map')
        region = Map.user_roi

    elif w_region.value == 'Input point and buffer':
        # Code cũ của bạn, giả sử w_point/w_buffer là global hoặc lấy từ children
        # Để an toàn, nên lấy từ w_region_detail.children
        children = w_region_detail.children
        if len(children) >= 2:
            w_point, w_buffer = children
            coord = w_point.value.split(',')
            coord = [float(a) for a in coord[:2]]
            region = ee.Geometry.Point(coord).buffer(w_buffer.value*1000)

    elif w_region.value == 'Rectangle from BBox':
        children = w_region_detail.children
        if len(children) >= 1:
            w_points = children[0]
            poly_coord = w_points.value.split(',')
            poly_coord = [float(a) for a in poly_coord]
            region = ee.Geometry.BBox(*poly_coord)

    elif w_region.value == 'Upload GeoJSON':
        if region_json is None:
            print("LỖI: Bạn chưa upload file GeoJSON hoặc quá trình upload chưa xong.")
            print("Vui lòng chọn file trong mục Region và đợi thông báo 'Đã upload thành công'.")
            return None
        
        try:
            print(f"Đang đọc file: {region_json}")
            region = geemap.geojson_to_ee(region_json)
        except Exception as e:
            print(f"Không thể chuyển đổi GeoJSON sang EE: {e}")
            return None

    elif w_region.value == 'Select an option':
        try:
            bounds = Map.get_bounds()
            region = ee.Geometry.BBox(*bounds)
        except:
            print("Không lấy được map bounds, vui lòng vẽ hoặc chọn vùng cụ thể.")
            region = None
            
    return region

### Cloud mask and image scaling


In [17]:
def maskL89C2clouds(image):
  """Cloud masking for Landsat 8/9 Collection 2 Level 2.
  Includes bit 4 (cloud shadow) for improved shadow detection.
  Bits: 0=Fill, 1=Dilated Cloud, 2=Cirrus, 3=Cloud, 4=Cloud Shadow
  """
  qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
  saturationMask = image.select('QA_RADSAT').eq(0)
  return image.updateMask(qaMask).updateMask(saturationMask) \
              .copyProperties(image, ['system:time_start'])

def maskL7C2clouds(image):
  """Cloud masking for Landsat 7 Collection 2 Level 2.
  Bits: 0=Fill, 1=Dilated Cloud, 3=Cloud, 4=Cloud Shadow
  Also applies SLC-off gap filling via focal mean for post-May 2003 imagery.
  """
  qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11011', 2)).eq(0)
  saturationMask = image.select('QA_RADSAT').eq(0)
  masked = image.updateMask(qaMask).updateMask(saturationMask)
  is_post_slc_failure = image.date().millis().gte(ee.Date('2003-05-31').millis())
  filled = masked.focalMean(1, 'square', 'pixels', 8)
  masked = ee.Image(ee.Algorithms.If(is_post_slc_failure, masked.unmask(filled), masked))
  return masked.copyProperties(image, ['system:time_start'])

def applyL89srScale(image):
  opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(opticalBands, None, True) \
              .addBands(thermalBands, None, True) \
              .copyProperties(image, ['system:time_start'])

applyLandsatSrScale = applyL89srScale

def maskS2CloudScorePlus(image, threshold=0.6):
  """CNN-based cloud/shadow masking using Google CloudScore+ for S2_HARMONIZED.
  Masks pixels where cs_cdf < threshold (default 0.6).
  The cs_cdf band must be joined to the image before calling this function.
  """
  cs_cdf = image.select('cs_cdf')
  mask = cs_cdf.gte(threshold)
  return image.updateMask(mask).divide(10000) \
              .copyProperties(image, ['system:time_start'])

def maskS2clouds(image):
  qa = image.select('QA60')
  cloudBitMask = 1 << 10
  cirrusBitMask = 1 << 11
  mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
      qa.bitwiseAnd(cirrusBitMask).eq(0))
  return image.updateMask(mask).divide(10000) \
              .copyProperties(image, ['system:time_start'])

### Filter ImageCollection


In [18]:
# Configurable cloud pixel percentage threshold for Sentinel-2 filtering
CLOUD_PIXEL_PCT_THRESHOLD = 20

def filterImageCollection(collection, region, start_date, end_date, bands, return_type='image'):
  """Return either a median composite image or an image collection.
  Routes S2_HARMONIZED to CloudScore+ masking, others to legacy masking.
  """

  image = ee.ImageCollection(collection)             .filterDate(start_date, end_date)

  if region is not None:
    image = image.filterBounds(region)

  if collection == 'COPERNICUS/S2_SR_HARMONIZED':
    image = image.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_PIXEL_PCT_THRESHOLD))
    # Join CloudScore+ cs_cdf band
    csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
    csFilter = ee.Filter.equals(leftField='system:index', rightField='system:index')
    image = image.linkCollection(csPlus, ['cs_cdf'])
    cs_threshold = w_cloudscore_threshold.value
    image = image.map(lambda img: maskS2CloudScorePlus(img, cs_threshold))
  elif collection == 'COPERNICUS/S2_HARMONIZED':
    image = image.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_PIXEL_PCT_THRESHOLD))
    image = image.map(maskS2clouds)
  elif collection.startswith('LANDSAT/L'):
    if collection == 'LANDSAT/LE07/C02/T1_L2':
      image = image.map(maskL7C2clouds)
    else:
      image = image.map(maskL89C2clouds)

  # For Landsat SR: apply scale factors
  if collection.endswith('T1_L2'):
    image = image.map(applyLandsatSrScale)

  # Check for empty collection after filtering
  if image.size().getInfo() == 0:
    print('Warning: No images found matching the specified filters (date/region/cloud). '
          'Please adjust your parameters.')
    return None

  image = image.select(bands)

  if return_type == 'image':
    image = image.median()
    if region is not None:
      if isinstance(region, ee.geometry.Geometry):
        image = image.clip(region)
      elif isinstance(region, ee.featurecollection.FeatureCollection):
        image = image.clipToCollection(region)
    return image
  else:
    if region is not None:
      if isinstance(region, ee.geometry.Geometry):
        image = image.map(lambda img: img.clip(region))
      elif isinstance(region, ee.featurecollection.FeatureCollection):
        image = image.map(lambda img: img.clipToCollection(region))
    return image


### Spectral indices/variables

```
satellite_list = [
    'Landsat 7 SR',
    'Landsat 8 SR',
    'Landsat 9 SR',
    'Landsat 7 TOA',
    'Landsat 8 TOA',
    'Landsat 9 TOA',
    'Sentinel-2 SR',
    'Sentinel-2 TOA']

collections_list = [
    'LANDSAT/LE07/C02/T1_L2',
    'LANDSAT/LC08/C02/T1_L2',
    'LANDSAT/LC09/C02/T1_L2',
    'LANDSAT/LE07/C02/T1_TOA',
    'LANDSAT/LC08/C02/T1_TOA',
    'LANDSAT/LC09/C02/T1_TOA',
    'COPERNICUS/S2_SR_HARMONIZED',
    'COPERNICUS/S2_HARMONIZED']
```

In [19]:
specialBandsNames = ['BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']
specialBandsList = [
  ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'],   # L7 SR
  ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'],   # L8 SR
  ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'],   # L9 SR
  ['B1','B2','B3','B4','B5','B7'],                     # L7 TOA
  ['B2','B3','B4','B5','B6','B7'],                     # L8 TOA
  ['B2','B3','B4','B5','B6','B7'],                     # L9 TOA
  ['B2','B3','B4','B8','B11','B12'],                   # S2 SR
  ['B2','B3','B4','B8','B11','B12'],                   # S2 TOA
]

specialBands = pd.DataFrame(np.array(specialBandsList),
                            columns=specialBandsNames)

def addSpectralIndices(image, ind, spectral_list):

  if 'None' in spectral_list:
    #print('No variables to be added.')
    return image

  #ind = collections_list.index(collection)
  b = specialBands.iloc[ind, :]
  #print('collection', collection, 'index', ind)
  #print('special bands name', b)

  #img_bands = image.bandNames().getInfo()
  img_bands = ee.List(image.bandNames())
  #print(img_bands)

  for var in spectral_list:
    if var == 'NBR':
      if img_bands.contains(b['SWIR2']) and img_bands.contains(b['NIR']):
        nbr = image.normalizedDifference([b['NIR'], b['SWIR2']]).rename('NBR')
        image = image.addBands(nbr)
      else:
        warnings.warn(f'Cannot compute NBR: missing required bands {b["SWIR2"]} and/or {b["NIR"]}')
    if var == 'EVI':
        if (img_bands.contains(b['NIR']) and img_bands.contains(b['RED'])
                and img_bands.contains(b['BLUE'])):
            evi = image.expression(
                '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
                {'NIR': image.select(b['NIR']),
                 'RED': image.select(b['RED']),
                 'BLUE': image.select(b['BLUE'])}
            ).rename('EVI')
            image = image.addBands(evi)
        else:
            warnings.warn('Cannot compute EVI: missing NIR/RED/BLUE bands')

    if var == 'MNDWI':
        if img_bands.contains(b['GREEN']) and img_bands.contains(b['SWIR1']):
            mndwi = image.normalizedDifference([b['GREEN'], b['SWIR1']]).rename('MNDWI')
            image = image.addBands(mndwi)
        else:
            warnings.warn('Cannot compute MNDWI: missing GREEN/SWIR1 bands')
    
    if var == 'NDBI':
        if img_bands.contains(b['SWIR1']) and img_bands.contains(b['NIR']):
            ndbi = image.normalizedDifference([b['SWIR1'], b['NIR']]).rename('NDBI')
            image = image.addBands(ndbi)
        else:
            warnings.warn('Cannot compute NDBI: missing SWIR1/NIR bands')
    if var == 'BSI':
        if (img_bands.contains(b['SWIR1']) and img_bands.contains(b['RED']) and
            img_bands.contains(b['NIR']) and img_bands.contains(b['BLUE'])):
            bsi = image.expression(
                '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))',
                {
                    'SWIR1': image.select(b['SWIR1']),
                    'RED': image.select(b['RED']),
                    'NIR': image.select(b['NIR']),
                    'BLUE': image.select(b['BLUE'])
                }
            ).rename('BSI')
            image = image.addBands(bsi)
        else:
            warnings.warn('Cannot compute BSI: missing SWIR1/RED/NIR/BLUE bands')
            
  return image

def addVariables(image, var_list, region=None):
  """Add other variables as predictors: topo, climatic, etc."""

  if not var_list:
    return image

  elevation = ee.Image('USGS/SRTMGL1_003').select('elevation')
  if region is not None:
    if isinstance(region, ee.geometry.Geometry):
      elevation = elevation.clip(region)
    elif isinstance(region, ee.featurecollection.FeatureCollection):
      elevation = elevation.clipToCollection(region)

  slope = ee.Terrain.slope(elevation)
  aspect = ee.Terrain.aspect(elevation)
  for var in var_list:
    if var == 'Elevation':
      image = image.addBands(elevation)
    if var == 'Slope':
      image = image.addBands(slope)
    if var == 'Aspect':
      image = image.addBands(aspect)
  return image

In [20]:
def getS1Composite(region, start_date, end_date, speckle_radius=50):
  """Acquire and preprocess Sentinel-1 GRD imagery.
  Filters to IW mode, descending orbit. Applies focal median speckle filter.
  Returns median composite with VV, VH, VV_VH_ratio bands.
  """
  s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
      .filterDate(start_date, end_date) \
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
      .filter(ee.Filter.eq('instrumentMode', 'IW')) \
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))

  if region is not None:
    if isinstance(region, ee.featurecollection.FeatureCollection):
      s1 = s1.filterBounds(region.geometry())
    else:
      s1 = s1.filterBounds(region)

  if s1.size().getInfo() == 0:
    print('Warning: No Sentinel-1 images found for the specified date range/region. Proceeding optical-only.')
    return None

  def speckle_filter(image):
    vv = image.select('VV').focal_median(speckle_radius, 'circle', 'meters')
    vh = image.select('VH').focal_median(speckle_radius, 'circle', 'meters')
    vv_vh_ratio = vv.subtract(vh).rename('VV_VH_ratio')
    return image.select([]).addBands([vv, vh, vv_vh_ratio]) \
               .copyProperties(image, ['system:time_start'])

  s1 = s1.map(speckle_filter)
  composite = s1.median()

  if region is not None:
    if isinstance(region, ee.geometry.Geometry):
      composite = composite.clip(region)
    elif isinstance(region, ee.featurecollection.FeatureCollection):
      composite = composite.clipToCollection(region)

  return composite


def getRawSARCollection():
  """Get raw Sentinel-1 image collection for temporal sampling.
  Dùng cho chế độ temporal (Yearly/Monthly/Daily) khi SAR Fusion được bật.
  Khác với getS1Composite() trả về 1 ảnh median, hàm này trả về toàn bộ
  collection đã qua speckle filter và gán date/month/year để join với optical.
  """
  s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
      .filterDate(start_date, end_date) \
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
      .filter(ee.Filter.eq('instrumentMode', 'IW')) \
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))

  if region is not None:
    if isinstance(region, ee.featurecollection.FeatureCollection):
      s1 = s1.filterBounds(region.geometry())
    else:
      s1 = s1.filterBounds(region)

  def speckle_filter(image):
    vv = image.select('VV').focal_median(w_speckle_radius.value, 'circle', 'meters')
    vh = image.select('VH').focal_median(w_speckle_radius.value, 'circle', 'meters')
    vv_vh_ratio = vv.subtract(vh).rename('VV_VH_ratio')
    return image.select([]).addBands([vv, vh, vv_vh_ratio]) \
               .copyProperties(image, ['system:time_start'])

  return s1.map(speckle_filter).map(addDates)

In [21]:
# === Harmonic Regression & Phenology ===
import math

def addHarmonicTerms(image, harmonics=2, include_trend=False):
  """Add harmonic (Fourier) independent variables to an image.

  Adds constant, optional linear trend, and cos/sin terms for each
  harmonic order.  All bands are cast to Float for numerical stability
  in the downstream linear regression.

  Args:
    image: ee.Image with a system:time_start property
    harmonics: int, number of harmonic orders (default 2)
    include_trend: bool, if True add a continuous time variable
        (fractional years since 1970-01-01) to capture inter-annual
        trends.  Recommended for multi-year time series where
        vegetation may be increasing or declining over time.

  Returns:
    ee.Image with added independent variable bands
  """
  doy = image.date().getRelative('day', 'year')
  t = ee.Number(doy).divide(365.25)

  independents = [ee.Image.constant(1).toFloat().rename('constant')]

  if include_trend:
    # Continuous time for inter-annual trend (fractional years since epoch)
    t_abs = image.date().difference(ee.Date('1970-01-01'), 'year')
    independents.append(ee.Image.constant(t_abs).toFloat().rename('t_trend'))

  for h in range(1, harmonics + 1):
    cos_term = t.multiply(2 * math.pi * h).cos()
    sin_term = t.multiply(2 * math.pi * h).sin()
    independents.append(ee.Image.constant(cos_term).toFloat().rename('cos' + str(h)))
    independents.append(ee.Image.constant(sin_term).toFloat().rename('sin' + str(h)))

  return image.addBands(ee.Image.cat(independents))


def _prefilter_vi_outliers(collection, dependent='EVI', low=-0.2, high=1.0):
  """Mask extreme vegetation index values before harmonic fitting.

  Removes pixels with VI outside [low, high] to reduce the influence of
  cloud/snow/shadow artifacts on OLS regression coefficients.

  The default range [-0.2, 1.0] covers the valid EVI range for
  surface-reflectance data (negative values can occur over water or
  shadow; values > 1.0 indicate sensor saturation or residual cloud).

  Args:
    collection: ee.ImageCollection containing the dependent band
    dependent: str, band name to filter (default 'EVI')
    low: float, minimum acceptable value (default -0.2)
    high: float, maximum acceptable value (default 1.0)

  Returns:
    ee.ImageCollection with outlier pixels masked
  """
  def _mask_outliers(image):
    vi = image.select(dependent)
    valid = vi.gte(low).And(vi.lte(high))
    return image.updateMask(valid)

  return collection.map(_mask_outliers)


def fitHarmonicRegression(collection, dependent='EVI', harmonics=2,
                          include_trend=False, filter_outliers=True):
  """Fit harmonic regression to a vegetation index time series.

  Model (without trend):
    VI(t) = a0 + sum_{h=1}^{H} [a_h cos(2 pi h t) + b_h sin(2 pi h t)]

  Model (with trend):
    VI(t) = a0 + beta1 * t_abs + sum_{h=1}^{H} [a_h cos(...) + b_h sin(...)]

  where t = DOY/365.25 (intra-annual) and t_abs = fractional years since
  1970-01-01 (inter-annual).

  Args:
    collection: ee.ImageCollection with the dependent band
    dependent: str, vegetation index band name (default 'EVI')
    harmonics: int, number of Fourier harmonics (default 2)
    include_trend: bool, add linear trend term (default False)
    filter_outliers: bool, pre-mask VI outliers (default True)

  Returns:
    ee.Image with regression coefficient bands + RMSE band
  """
  if filter_outliers:
    collection = _prefilter_vi_outliers(collection, dependent)

  independentNames = ['constant']
  if include_trend:
    independentNames.append('t_trend')
  for h in range(1, harmonics + 1):
    independentNames.extend(['cos' + str(h), 'sin' + str(h)])

  harmonicCollection = collection.map(
      lambda img: addHarmonicTerms(img, harmonics, include_trend)
  )

  harmonicTrend = harmonicCollection.select(independentNames + [dependent]) \
      .reduce(ee.Reducer.linearRegression(
          numX=len(independentNames),
          numY=1
      ))

  coefficients = harmonicTrend.select('coefficients') \
      .arrayProject([0]).arrayFlatten([independentNames])

  residuals = harmonicTrend.select('residuals').arrayGet([0]).rename('RMSE')

  return coefficients.addBands(residuals)


def extractHarmonicFeatures(coefficients, dependent='EVI', harmonics=2):
  """Compute amplitude, phase, and mean from harmonic regression coefficients."""
  mean = coefficients.select('constant').rename(dependent + '_mean')

  features = [mean]
  for h in range(1, harmonics + 1):
    cos_coeff = coefficients.select('cos' + str(h))
    sin_coeff = coefficients.select('sin' + str(h))

    amplitude = cos_coeff.pow(2).add(sin_coeff.pow(2)).sqrt() \
        .rename(dependent + '_amp' + str(h))
    phase = sin_coeff.atan2(cos_coeff).rename(dependent + '_phase' + str(h))

    features.extend([amplitude, phase])

  return ee.Image.cat(features)


def extractPhenologyMetrics(harmonicImage, dependent='EVI', threshold_fraction=0.2):
  """Extract SoS, EoS, and GSL from first-harmonic regression coefficients.

  Model: VI(t) = a0 + A * cos(2*pi*t/T - phase)
    - Peak at t_peak = phase * T / (2*pi)
    - Base = a0 - A,  Peak = a0 + A,  Range = 2*A

  SoS/EoS are defined where the fitted curve crosses:
    level = base + threshold_fraction * range
          = (a0 - A) + threshold_fraction * 2*A
          = a0 + (2*threshold_fraction - 1) * A

  This means: cos(theta) = 2*threshold_fraction - 1

  Default threshold_fraction=0.2 follows Jonsson & Eklundh (2004) TIMESAT:
    cos(theta) = 2*0.2 - 1 = -0.6  =>  theta = acos(-0.6) ~ 2.214 rad
    GSL ~ 258 days (typical for temperate regions)

  White et al. (2009) use threshold_fraction=0.5:
    cos(theta) = 0  =>  theta = pi/2
    GSL ~ 182 days

  Note: Only the first harmonic (h=1, annual cycle) is used for phenology
  extraction, even when higher harmonics are fitted. This is standard
  practice as SoS/EoS represent the fundamental annual growing season
  (Jonsson & Eklundh 2004).

  Args:
    harmonicImage: ee.Image with '{dep}_phase1', '{dep}_amp1', '{dep}_mean'
    dependent: str, vegetation index name (default 'EVI')
    threshold_fraction: float in (0, 1), fraction of seasonal amplitude
        above base level to define SoS/EoS.
        0.2 = TIMESAT default (Jonsson & Eklundh 2004)
        0.5 = 50% amplitude (White et al. 2009)

  Returns:
    ee.Image with bands: SoS_doy, EoS_doy, GSL_days
  """
  phase1 = harmonicImage.select(dependent + '_phase1')
  amp1 = harmonicImage.select(dependent + '_amp1')

  # cos(theta) = 2 * threshold_fraction - 1
  cos_theta = 2.0 * threshold_fraction - 1.0
  threshold_angle = ee.Image.constant(cos_theta).acos()

  # SoS: ascending limb (before peak)
  sos_rad = phase1.subtract(threshold_angle)
  sos_doy = sos_rad.multiply(365.25 / (2 * math.pi)).mod(365.25).add(1).rename('SoS_doy')

  # EoS: descending limb (after peak)
  eos_rad = phase1.add(threshold_angle)
  eos_doy = eos_rad.multiply(365.25 / (2 * math.pi)).mod(365.25).add(1).rename('EoS_doy')

  # Growing season length with year-boundary wrapping
  gsl = eos_doy.subtract(sos_doy)
  gsl = gsl.where(gsl.lt(0), gsl.add(365)).rename('GSL_days')

  # --- Edge case handling ---
  # Clamp GSL to valid range [0, 365]
  gsl = gsl.min(365).max(0)

  # Mask pixels with negligible amplitude (evergreen / bare / water)
  # Amplitude < 0.01 in EVI reflectance units means no clear seasonality
  valid_season = amp1.gt(0.01)
  sos_doy = sos_doy.updateMask(valid_season)
  eos_doy = eos_doy.updateMask(valid_season)
  gsl = gsl.updateMask(valid_season)

  return ee.Image.cat([sos_doy, eos_doy, gsl])


### Get image

In [22]:
def getImage():
    """Build predictor image from widgets: optical composite / harmonic features / topo vars / SAR fusion."""
    global region, start_date, end_date
    global feature_names, bands, spectral_list, var_list
    global split_ratio, seed, scale

    seed = 37643

    def _resolve_dates():
        if w_start_date.value is None or w_end_date.value is None:
            print('Error: Please select start and end dates.')
            return None, None

        start_str = w_start_date.value.strftime('%Y-%m-%d')
        end_str = w_end_date.value.strftime('%Y-%m-%d')

        if end_str <= start_str:
            print('Error: End date must be after start date.')
            return None, None

        return start_str, end_str

    def _resolve_scale(collection_ids):
        if any(c.startswith('COPERNICUS') for c in collection_ids):
            return 10
        if any(c.startswith('LANDSAT') for c in collection_ids):
            return 30
        return 30

    def _resolve_vis_bands(band_names):
        candidates = [
            ['B4', 'B3', 'B2'],
            ['SR_B4', 'SR_B3', 'SR_B2'],
            ['RED', 'GREEN', 'BLUE'],
        ]
        for candidate in candidates:
            if all(b in band_names for b in candidate):
                return candidate
        return None

    def _build_single_source_image(ind, collection, selected_bands):
        nonlocal image_median_vis

        if w_harmonic_regression.value:
            print(f'Fitting harmonic regression (order={w_harmonic_order.value})...')

            img_col = filterImageCollection(
                collection, region, start_date, end_date, selected_bands, return_type='collection'
            )
            if img_col is None:
                return None

            img_col = img_col.map(lambda img: addSpectralIndices(img, ind, spectral_list))

            dependent = 'EVI'
            

            coefficients = fitHarmonicRegression(img_col, dependent, w_harmonic_order.value)
            harmonic_features = extractHarmonicFeatures(coefficients, dependent, w_harmonic_order.value)

            image = harmonic_features

            if w_phenology_metrics.value:
                print('Extracting phenology metrics (SoS, EoS, GSL)...')
                pheno_metrics = extractPhenologyMetrics(harmonic_features, dependent)
                image = image.addBands(pheno_metrics)

            image_median_vis = img_col.median()
            if region is not None:
                if isinstance(region, ee.geometry.Geometry):
                    image_median_vis = image_median_vis.clip(region)
                elif isinstance(region, ee.featurecollection.FeatureCollection):
                    image_median_vis = image_median_vis.clipToCollection(region)

            return image

        image = filterImageCollection(collection, region, start_date, end_date, selected_bands)
        if image is None:
            return None

        image = addSpectralIndices(image, ind, spectral_list)
        image_median_vis = image
        return image

    def _build_multi_source_image():
        nonlocal image_median_vis

        image = ee.Image()
        collection_ids = []

        for sat, sat_bands in multiSource.items():
            ind = satellite_list.index(sat)
            collection = collections_list[ind]
            collection_ids.append(collection)

            image0 = filterImageCollection(collection, region, start_date, end_date, sat_bands)
            if image0 is None:
                continue

            image0 = addSpectralIndices(image0, ind, spectral_list)
            image = image.addBands(image0)

        image_median_vis = image
        return image, collection_ids

    start_end = _resolve_dates()
    if start_end == (None, None):
        return None
    start_date, end_date = start_end

    region = getRegion()
    if region is None:
        print('No region defined. Please draw a shape, enter coordinates, or upload a GeoJSON file.')
        return None

    split_ratio = w_split_ratio.value
    spectral_list = list(w_spectral_indices.value)
    var_list = list(w_variables.value)

    image_median_vis = None

    if (w_multiSource.value is False) or not multiSource:
        ind = satellite_list.index(w_satellite.value)
        collection = collections_list[ind]
        selected_bands = list(w_select_bands.value)

        print('Collection:', collection)
        image = _build_single_source_image(ind, collection, selected_bands)
        if image is None:
            return None

        scale = _resolve_scale([collection])

    else:
        print('Multi-source selected.')
        image, used_collections = _build_multi_source_image()
        scale = _resolve_scale(used_collections)

    image = addVariables(image, var_list, region)

    if w_sar_fusion.value:
        print('Fusing Sentinel-1 SAR data...')
        s1_composite = getS1Composite(region, start_date, end_date, w_speckle_radius.value)
        if s1_composite is not None:
            image = image.addBands(s1_composite)
            print('SAR bands added: VV, VH, VV_VH_ratio')
        else:
            print('SAR fusion skipped (no S1 data).')

    feature_names = image.bandNames().getInfo()
    if 'constant' in feature_names:
        feature_names.remove('constant')
        image = image.select(feature_names)

    bands = feature_names
    print('Bands/feature names:', feature_names)

    try:
        if image_median_vis is not None:
            img_bands_info = image_median_vis.bandNames().getInfo()
            vis_bands = _resolve_vis_bands(img_bands_info)

            if vis_bands is not None:
                vis_params = {'min': 0, 'max': 0.3, 'bands': vis_bands}
                Map.addLayer(image_median_vis, vis_params, 'Optical composite')
                with w_map_output:
                    text = f'Added image layers (natural color {vis_bands}).'
                    display(widgets.HTML(value=f"{text}"))
            else:
                print(f'Lưu ý: Không thể hiển thị bản đồ vì thiếu band RGB phù hợp. Các band hiện có: {img_bands_info}')
    except Exception as e:
        print(f'Lỗi khi hiển thị bản đồ Optical composite: {str(e)}')

    if region is not None:
        Map.centerObject(region, 12)

    return image


### Get temporal composites

In [23]:
def addDates(img):
  """Add properties: date, month, year, or even doy and week """
  date = img.date()
  date_str = date.format('YYYY-MM-dd')
  month_str = date.format('YYYY-MM')
  return img.set('date', date_str) \
            .set('month', month_str) \
            .set('year', date.get('year'))

def temporalComposite(dataset, interval):
  """create temporal composite for classifications
  by one of GEE join: date_match, month_match, year_match

  date_match is also used to get sample features if there is a 'date' property
  associated with labeled points
  """

  temporal_match = interval + '_match'

  distinctDates = dataset.distinct(interval).sort(interval)
  filter = ee.Filter.equals(leftField=interval, rightField=interval)
  join = ee.Join.saveAll(temporal_match)
  joinCol = join.apply(distinctDates, dataset, filter)

  def mapJoinCollection(img):
    tmp = ee.ImageCollection.fromImages(img.get(temporal_match))

    if interval == 'date':
      tmp = tmp.mosaic()
    else:
      tmp = tmp.median()

    return tmp.copyProperties(img, ['date', 'month', 'year', 'system:time_start'])

  colMosaic = joinCol.map(mapJoinCollection)

  return colMosaic

def getRawImageCollections():
  """Get raw image collection, which is later used to create temporal compoiste

  Does not yet support multi-source
  """
  # dataset = getRawImageCollections()
  if w_multiSource.value == False or not multiSource:
    ind = satellite_list.index(w_satellite.value)
    collection = collections_list[satellite_list.index(w_satellite.value)]
    bands = list(w_select_bands.value)

    dataset = filterImageCollection(collection, region, start_date, end_date, bands,
                                    return_type='collection')
    dataset = dataset.map(addDates)

    dataset = dataset.map(lambda img: addSpectralIndices(img, ind, spectral_list))
    dataset = dataset.map(lambda img: addVariables(img, var_list, region))

    return dataset
  else:
    raise Exception("No support yet for multi-source composite for spatiotemporal dynamics.")


### Sample collections by feature dates

In [24]:
def sampleRegionsByTemporalUnit(colMosaic_unit, unit='date'):
  """Sample temporal composite for predictor variables according to a temporal unit.
     unit ∈ {'date', 'month', 'year'}
  """

  match_prop = unit
  match_name = unit + '_match'

  filter0 = ee.Filter.equals(leftField=match_prop, rightField=match_prop)
  join0 = ee.Join.saveAll(match_name)

  labels_distinct = LABEL_DATA.filter(
      ee.Filter.notNull([match_prop])
  ).distinct(match_prop).sort(match_prop)

  joinCol_0 = join0.apply(labels_distinct, colMosaic_unit, filter0)

  def mapSampleRegions(feat):
    matched_list = ee.List(feat.get(match_name))

    return ee.Algorithms.If(
        matched_list.size().gt(0),
        ee.FeatureCollection(
            ee.Image(matched_list.get(0)).sampleRegions(
                collection=LABEL_DATA.filter(ee.Filter.eq(match_prop, feat.get(match_prop))),
                properties=[LABEL],
                tileScale=4,
                scale=scale,
                geometries=True
            )
        ),
        ee.FeatureCollection([])
    )

  print(f'Sample temporal composite according to feature collections matched by {unit} ...')
  samples = ee.FeatureCollection(joinCol_0.map(mapSampleRegions).flatten())

  return samples

In [25]:
# getTemporalCompSamples removed: logic replaced by getSamplesData

### Get samples data

In [26]:
# === Spatial Block Cross-Validation ===

def generateSpatialGrid(roi, blockSize_km=10, nFolds=5):
  """Generate a regular grid of rectangular blocks over the ROI.
  Fold assignment uses horizontal strips so each fold is a contiguous
  spatial band — train/test zones are fully separated (no checkerboard).
  Block width is corrected for latitude: 1 deg lon = cos(lat)*111 km.

  Args:
    roi: ee.Geometry or ee.FeatureCollection
    blockSize_km: float, block size in km
    nFolds: int, number of CV folds
  Returns:
    ee.FeatureCollection of grid blocks with 'fold' property
  """
  import math
  if isinstance(roi, ee.featurecollection.FeatureCollection):
    geom = roi.geometry()
  else:
    geom = roi

  # Get bounding box
  bounds = geom.bounds()
  coords = ee.List(bounds.coordinates().get(0))

  xMin = ee.Number(ee.List(coords.get(0)).get(0))
  yMin = ee.Number(ee.List(coords.get(0)).get(1))
  xMax = ee.Number(ee.List(coords.get(2)).get(0))
  yMax = ee.Number(ee.List(coords.get(2)).get(1))

  # Latitude-corrected block size in degrees
  # 1 deg latitude  ~ 111.0 km (uniform)
  # 1 deg longitude ~ cos(lat_center) * 111.0 km
  lat_center_rad = yMin.add(yMax).divide(2).multiply(math.pi / 180)
  blockSize_lat_deg = blockSize_km / 111.0
  blockSize_lon_deg = ee.Number(blockSize_km).divide(
      ee.Number(111.0).multiply(lat_center_rad.cos())
  )

  xSteps = xMax.subtract(xMin).divide(blockSize_lon_deg).ceil().int()
  ySteps = yMax.subtract(yMin).divide(blockSize_lat_deg).ceil().int()

  # Strip-based fold: all blocks in same Y-row share the same fold ID
  # → each fold is a contiguous horizontal band across the study area
  def makeRow(yIdx):
    yIdx = ee.Number(yIdx)
    y0 = yMin.add(yIdx.multiply(blockSize_lat_deg))
    y1 = y0.add(blockSize_lat_deg)
    fold = yIdx.mod(nFolds)

    def makeBlock(xIdx):
      xIdx = ee.Number(xIdx)
      x0 = xMin.add(xIdx.multiply(blockSize_lon_deg))
      x1 = x0.add(blockSize_lon_deg)
      block = ee.Feature(ee.Geometry.Rectangle([x0, y0, x1, y1]))
      return block.set('fold', fold)

    return ee.List.sequence(0, xSteps.subtract(1)).map(makeBlock)

  blocks = ee.FeatureCollection(
      ee.List.sequence(0, ySteps.subtract(1)).map(makeRow).flatten()
  )

  # Filter to blocks that intersect the ROI
  blocks = blocks.filterBounds(geom)

  return blocks


def spatialBlockCV(samples, grid, nFolds, classifierFn, featureNames, labelField):
  """Run k-fold spatial block cross-validation.

  Args:
    samples: ee.FeatureCollection with features and labels
    grid: ee.FeatureCollection of spatial blocks with 'fold' property
    nFolds: int
    classifierFn: function that returns an untrained ee.Classifier
    featureNames: list of band/feature names
    labelField: str, label property name
  Returns:
    list of dicts with per-fold metrics
  """
  fold_results = []

  for fold in range(nFolds):
    print(f'  Fold {fold + 1}/{nFolds}...')

    # Get test blocks for this fold
    test_blocks = grid.filter(ee.Filter.eq('fold', fold))
    train_blocks = grid.filter(ee.Filter.neq('fold', fold))

    # Spatial join: assign samples to blocks
    spatialFilter = ee.Filter.intersects(leftField='.geo', rightField='.geo')

    train_samples = ee.Join.simple().apply(samples, train_blocks, spatialFilter)
    test_samples = ee.Join.simple().apply(samples, test_blocks, spatialFilter)

    train_size = train_samples.size().getInfo()
    test_size = test_samples.size().getInfo()

    if train_size == 0 or test_size == 0:
      print(f'    Skipping fold {fold + 1}: train={train_size}, test={test_size}')
      continue

    # Train classifier
    clf = classifierFn()
    clf = clf.train(train_samples, labelField, featureNames)

    # Classify test samples
    test_classified = test_samples.classify(clf)

    # Get results
    result_df = geemap.ee_to_df(test_classified.select([labelField, 'classification']))
    y_true = result_df[labelField].values
    y_pred = result_df['classification'].values

    from sklearn.metrics import (accuracy_score, cohen_kappa_score, precision_score, recall_score, f1_score)

    oa = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)

    n_classes = len(CLASS_NAMES)
    labels = list(range(n_classes))
    ua = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    pa = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)

    # Quantity & Allocation Disagreement (Pontius & Millones 2011)
    cm_fold = confusion_matrix(y_true, y_pred, labels=labels)
    cm_prop_fold = cm_fold.astype(float) / cm_fold.sum()
    qd = np.sum(np.abs(cm_prop_fold.sum(axis=1) - cm_prop_fold.sum(axis=0))) / 2
    ad = (1 - oa) - qd

    fold_results.append({
        'fold': fold,
        'oa': oa,
        'kappa': kappa,
        'ua': ua,
        'pa': pa,
        'qd': qd,
        'ad': ad,
        'train_size': train_size,
        'test_size': test_size,
        'y_true': y_true,
        'y_pred': y_pred
    })

    print(f'    OA={oa:.3f}, Kappa={kappa:.3f}, QD={qd:.3f}, AD={ad:.3f}, train={train_size}, test={test_size}')

  return fold_results

def aggregateCVResults(foldResults, verbose=True):
  """Aggregate spatial block CV results across folds.

  Returns:
    dict with mean/std for OA, kappa, and per-class UA/PA
  """
  oas = [r['oa'] for r in foldResults]
  kappas = [r['kappa'] for r in foldResults]
  uas = np.array([r['ua'] for r in foldResults])
  pas = np.array([r['pa'] for r in foldResults])

  summary = {
      'mean_oa': np.mean(oas),
      'std_oa': np.std(oas),
      'mean_kappa': np.mean(kappas),
      'std_kappa': np.std(kappas),
      'mean_ua': np.mean(uas, axis=0),
      'std_ua': np.std(uas, axis=0),
      'mean_pa': np.mean(pas, axis=0),
      'std_pa': np.std(pas, axis=0),
      'mean_qd': np.mean([r['qd'] for r in foldResults]),
      'std_qd': np.std([r['qd'] for r in foldResults]),
      'mean_ad': np.mean([r['ad'] for r in foldResults]),
      'std_ad': np.std([r['ad'] for r in foldResults]),
  }
  if verbose:
    print(f'\nSpatial Block CV Summary ({len(foldResults)} folds):')
    print(f'  Overall Accuracy: {summary["mean_oa"]:.3f} +/- {summary["std_oa"]:.3f}')
    print(f'  Kappa: {summary["mean_kappa"]:.3f} +/- {summary["std_kappa"]:.3f}')
    print(f'  Quantity Disagreement: {summary["mean_qd"]:.4f} +/- {summary["std_qd"]:.4f}')
    print(f'  Allocation Disagreement: {summary["mean_ad"]:.4f} +/- {summary["std_ad"]:.4f}')
    print(f'  Per-class User\'s Accuracy (mean +/- std):')
    for i, name in enumerate(CLASS_NAMES):
      print(f'    {name}: {summary["mean_ua"][i]:.3f} +/- {summary["std_ua"][i]:.3f}')
    print(f'  Per-class Producer\'s Accuracy (mean +/- std):')
    for i, name in enumerate(CLASS_NAMES):
      print(f'    {name}: {summary["mean_pa"][i]:.3f} +/- {summary["std_pa"][i]:.3f}')

  return summary


In [27]:
def getSamplesData(imageM):
    """..."""
    global CLASS_NAMES, LABEL_DATA, CLASS_PALETTE, temporal_interval
    global training, testing, samples_all, evaluation_mode
    global colMosaic_date, colMosaic_tmp
    global spatial_grid, cv_fold_results

    if w_sample_select.value == 'Upload samples':
        res = getClassLabelsFromFile()
        if isinstance(res, tuple) and len(res) == 2:
            CLASS_NAMES, LABEL_DATA = res

        if CLASS_NAMES is None or LABEL_DATA is None:
            print('Error: Could not read class labels from file. Please check your upload.')
            return

        if (w_randomColors.value is False) and (len(w_palette.children) > 0):
            CLASS_PALETTE = [c.value for c in w_palette.children]
        else:
            if CLASS_PALETTE is None or len(CLASS_PALETTE) != len(CLASS_NAMES):
                CLASS_PALETTE = [
                    "#" + "".join([random.choice("0123456789ABCDEF") for _ in range(6)])
                    for _ in range(len(CLASS_NAMES))
                ]
        print('Labels read from file.')

    elif w_sample_select.value == 'Sample LULC products':
        LABEL_DATA, CLASS_PALETTE = getClassLabelsFromLC()
        print('No date assigned to LABEL_DATA.')

    if w_temporal.value == 'All dates':
        temporal_interval = 'all-time'
        print("No temporal composite for all-time interval. Samples are drawn from the all-time predictor image.")
    elif w_temporal.value == 'Yearly':
        temporal_interval = 'year'
    elif w_temporal.value == 'Monthly':
        temporal_interval = 'month'
    elif w_temporal.value == 'Daily':
        temporal_interval = 'date'

    has_date_col = 'date' in LABEL_DATA.first().propertyNames().getInfo()

    # ── Bước 1: Build temporal composite (chỉ khi cần) ──────────────────────
    def get_synced_composite(interval_str):
        dataset_opt = getRawImageCollections()
        dataset_sar = None
        if w_sar_fusion.value:
            dataset_sar = getRawSARCollection()

        comp_opt = temporalComposite(dataset_opt, interval_str)

        if dataset_sar is not None:
            comp_sar = temporalComposite(dataset_sar, interval_str)

            if interval_str == 'date':
                max_diff = 3 * 24 * 60 * 60 * 1000
                time_filter = ee.Filter.maxDifference(
                    difference=max_diff,
                    leftField='system:time_start',
                    rightField='system:time_start'
                )
                join = ee.Join.saveBest('best_sar', 'time_dist')
                joined = join.apply(comp_opt, comp_sar, time_filter)
                return joined.map(lambda f: ee.Image(f).addBands(ee.Image(f.get('best_sar'))))
            else:
                time_filter = ee.Filter.equals(leftField=interval_str, rightField=interval_str)
                joined = ee.ImageCollection(ee.Join.inner().apply(comp_opt, comp_sar, time_filter))
                return joined.map(lambda f: ee.Image(f.get('primary')).addBands(ee.Image(f.get('secondary'))))

        return comp_opt

    if temporal_interval != 'all-time':
        colMosaic_tmp = get_synced_composite(temporal_interval)
    elif has_date_col:
        # all-time nhưng sample có cột date → cần composite theo ngày để sample
        pass  # colMosaic_date sẽ được build bên dưới

    # ── Bước 2: Sample (LUÔN chạy, bất kể temporal_interval) ────────────────
    print(f'Sample regions for predictor variables, scale={scale} ...')

    if (not has_date_col) or (temporal_interval == 'all-time'):
    # Không có date trong nhãn, hoặc user chọn All dates:
        labels_in_roi = LABEL_DATA.filterBounds(region)

        print('Filtering labels by ROI before sampling...')
        print('Labels inside ROI:', labels_in_roi.size().getInfo())

        samples = imageM.sampleRegions(
            collection=labels_in_roi,
            properties=[LABEL],
            scale=scale,
            tileScale=8,
            geometries=True
        )
    else:
        # Daily / Monthly / Yearly với sample có date
        if temporal_interval == 'date':
            sample_unit = 'date'
            colMosaic_unit = colMosaic_tmp
        elif temporal_interval == 'month':
            sample_unit = 'month'
            colMosaic_unit = colMosaic_tmp
        elif temporal_interval == 'year':
            sample_unit = 'year'
            colMosaic_unit = colMosaic_tmp
        else:
            sample_unit = 'date'
            colMosaic_unit = get_synced_composite('date')

        samples = sampleRegionsByTemporalUnit(colMosaic_unit, sample_unit)

    samples_all = samples

    # ── Bước 3: Phân chia train/test ────────────────────────────────────────
    if w_spatial_block_cv.value:
        evaluation_mode = 'spatial_cv'
        print(f'Generating spatial grid (block size={w_block_size_km.value} km, folds={w_n_folds.value})...')
        spatial_grid = generateSpatialGrid(region, w_block_size_km.value, w_n_folds.value)

        training = samples_all
        testing = None

        samples_with_geo = samples_all.map(lambda f: f.setGeometry(f.geometry()))

        print('Running spatial block cross-validation...')

        def make_classifier():
            if w_classifier_customize.value:
                param_dict = {w.description: w.value for w in w_classifier_param.children}
                return ee.Classifier.smileRandomForest(**param_dict, seed=seed)
            else:
                return ee.Classifier.smileRandomForest(100)

        cv_fold_results = spatialBlockCV(
            samples_with_geo,
            spatial_grid,
            w_n_folds.value,
            make_classifier,
            feature_names,
            LABEL
        )

        if cv_fold_results:
            aggregateCVResults(cv_fold_results)
            print('Note: Spatial Block CV metrics are from out-of-fold predictions.')
            print('Final map classifier will use ALL training samples (standard practice, see Wainer & Cawley 2021).')

    else:
        evaluation_mode = 'holdout'
        samples_all = samples_all.randomColumn(seed=seed)
        training = samples_all.filter(ee.Filter.lt('random', split_ratio))
        testing = samples_all.filter(ee.Filter.gte('random', split_ratio))
        cv_fold_results = None
        spatial_grid = None

    # ── Bước 4: Visualize ───────────────────────────────────────────────────
    train_styled = training.select([LABEL]).map(lambda f: f.set('style', {
        'color': ee.List(CLASS_PALETTE).get(f.get(LABEL)),
        'pointShape': 'circle',
        'pointSize': 6
    }))

    Map.centerObject(region, 10)
    layers_to_remove = ['Train labels', 'Test labels', 'Classified (median)', 'Spatial CV Grid']
    for layer in list(Map.layers):
        if getattr(layer, 'name', '') in layers_to_remove:
            Map.remove_layer(layer)
    train_preview = train_styled.limit(300)
    Map.addLayer(train_preview.style(styleProperty='style'), {}, 'Train labels preview')

    if spatial_grid is not None:
        Map.addLayer(spatial_grid, {'color': 'blue'}, 'Spatial CV Grid')

    if (evaluation_mode == 'holdout') and (testing is not None):
        test_styled = testing.select([LABEL]).map(lambda f: f.set('style', {
            'color': ee.List(CLASS_PALETTE).get(f.get(LABEL)),
            'pointShape': 'triangle',
            'pointSize': 8
        }))
        Map.addLayer(test_styled.style(styleProperty='style'), {}, 'Test labels')

    w_map_output.clear_output()
    with w_map_output:
        if evaluation_mode == 'spatial_cv':
            display(widgets.HTML(value='Spatial Block CV mode: map is trained on all samples; accuracy is evaluated only from out-of-fold predictions.'))
        else:
            display(widgets.HTML(value='Holdout mode: Train/Test labels added.'))
    return training

### GEE classifier

In [28]:
def runGeeClassifier(imageM, training):
    """Train GEE classifier and classify the image."""
    global classifier, classifier_param_dict, image_classified, resultVis

    if imageM is None:
        print('Error: imageM is None. Please generate image first.')
        return None

    if training is None:
        print('Error: training data is missing. Please run getSamplesData() first.')
        return None

    if not feature_names:
        print('Error: feature_names is empty.')
        return None

    if CLASS_NAMES is None or CLASS_PALETTE is None:
        print('Error: CLASS_NAMES / CLASS_PALETTE is missing.')
        return None

    print('Train GEE classifier ...')

    classifier_select = w_classifier_select.value
    classifier_param_dict = {}

    if classifier_select != 'RandomForest':
        print(f'Classifier "{classifier_select}" is not supported yet.')
        return None

    if w_classifier_customize.value:
        classifier_param_dict = {w.description: w.value for w in w_classifier_param.children}
        print('Using customized parameters:', classifier_select)
        classifier = ee.Classifier.smileRandomForest(
            **classifier_param_dict,
            seed=seed
        )
    else:
        print('Using default parameters:', classifier_select)
        classifier_param_dict = {'numberOfTrees': 100}
        classifier = ee.Classifier.smileRandomForest(100)

    classifier = classifier.train(
        features=training,
        classProperty=LABEL,
        inputProperties=feature_names
    )

    print('Classify the image ...')

    if temporal_interval == 'all-time':
        classify_target = imageM.select(feature_names)
    else:
        if colMosaic_tmp is None:
            print('Error: Temporal composite is missing.')
            return None
        classify_target = ee.ImageCollection(colMosaic_tmp).median().select(feature_names)

    image_classified = classify_target.classify(classifier)

    resultVis = {
        'min': 0,
        'max': len(CLASS_NAMES) - 1,
        'palette': CLASS_PALETTE
    }

    layers_to_remove = ['Classified (median)']
    for layer in list(Map.layers):
        if getattr(layer, 'name', '') in layers_to_remove:
            Map.remove_layer(layer)

    Map.addLayer(image_classified, resultVis, 'Classified (median)')

    Map.remove_legends()
    legend_dict = dict(zip(CLASS_NAMES, CLASS_PALETTE))
    Map.add_legend(
        legend_dict=legend_dict,
        title='Legend',
        position='bottomright'
    )

    with w_map_output:
        w_map_output.clear_output()
        display(widgets.HTML(value='Added classified image with legend.'))

    return image_classified


### Sklearn classifer & metrics


In [29]:


def classifierMetrics(y_test, y_pred):
  """Print classification report, accuracy_score, kappa coefficient,
  quantity/allocation disagreement (Pontius & Millones 2011),
  and confusion matrix.
  """
  labels = list(range(0, len(CLASS_NAMES)))
  print('Classification report:')
  print(classification_report(y_test, y_pred, labels=labels, target_names=CLASS_NAMES))

  oa = accuracy_score(y_test, y_pred)
  kappa = cohen_kappa_score(y_test, y_pred)
  cm = confusion_matrix(y_test, y_pred, labels=labels)

  # Quantity & Allocation Disagreement (Pontius & Millones 2011)
  cm_prop = cm.astype(float) / cm.sum()
  qd = np.sum(np.abs(cm_prop.sum(axis=1) - cm_prop.sum(axis=0))) / 2
  ad = (1 - oa) - qd

  print('Accuracy score: {:.2f}'.format(oa))
  print('Kappa coefficient: {:.2f}'.format(kappa))
  print('Quantity disagreement: {:.4f}'.format(qd))
  print('Allocation disagreement: {:.4f}'.format(ad))

  print('Confusion matrix:')
  print(cm)
  print('')

def sklearnClassifier(training, testing):
    global classifier_param_dict

    train_data = geemap.ee_to_df(training)
    test_data = geemap.ee_to_df(testing)

    X_train = train_data[feature_names]
    y_train = train_data[LABEL]

    X_test = test_data[feature_names]
    y_test = test_data[LABEL]

    if not w_classifier_customize.value:
        classifier_param_dict = {
            'numberOfTrees': 100,
            'minLeafPopulation': 5
        }
    else:
        classifier_param_dict = {w.description: w.value for w in w_classifier_param.children}

    n_estimators = int(classifier_param_dict.get('numberOfTrees', 100))
    min_samples_leaf = int(classifier_param_dict.get('minLeafPopulation', 1))

    forest = RandomForestClassifier(
        n_estimators=n_estimators,
        criterion='gini',
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=min_samples_leaf,
        max_features='sqrt',
        bootstrap=True,
        random_state=seed,
        n_jobs=-1
    )

    forest.fit(X_train, y_train)

    

    return forest, X_train, X_test, y_train, y_test

### Zonal statistics

In [30]:
def resolve_analysis_scale(image=None, fallback=None):
    """
    Resolve analysis scale automatically for reduction/sampling.

    Priority:
    1) global `scale` already resolved in getImage()
    2) current satellite widget (w_satellite)
    3) image native nominal scale (if available)
    4) fallback
    """
    # 1) Use global scale if getImage() has already set it
    try:
        if isinstance(scale, (int, float)) and scale > 0:
            return int(scale)
    except Exception:
        pass

    # 2) Use satellite widget
    try:
        sat = str(w_satellite.value).lower()
        if 'sentinel-2' in sat or 'sentinel 2' in sat or 's2' in sat:
            return 10
        if 'landsat' in sat:
            return 30
    except Exception:
        pass

    # 3) Fallback to image nominal scale
    if image is not None:
        try:
            img = ee.Image(image)
            first_band = ee.String(img.bandNames().get(0))
            nominal = img.select([first_band]).projection().nominalScale().getInfo()
            if nominal is not None:
                nominal = float(nominal)
                # snap to common sensor scales for stability/reporting
                if nominal <= 15:
                    return 10
                elif nominal <= 40:
                    return 30
                return nominal
        except Exception:
            pass

    # 4) Final fallback
    if fallback is not None:
        return fallback

    return 30

In [31]:
def zonalStatistics(classified, roi=None, return_type='dict'):
  """Calculate zonal areas for each class, given classified image

  Args:
    classified: classified image with band 'classification'
    return_type: 'dict' or 'feature', returns a dict or ee.Feature
  """

  if roi == None:
    roi = region

  # Calculate areas (km^2) for each class
  areaImage = ee.Image.pixelArea().divide(1e6).addBands(classified)

  areas = areaImage.reduceRegion(
              **{
                  "reducer": ee.Reducer.sum().group(
                      **{
                          "groupField": 1,
                          "groupName": "classification",
                      }
                  ),
                  "geometry": roi,
                  "scale": 50,
                  "maxPixels": 1e13,
                  "bestEffort": False,
                  "tileScale": 8
              }
  )

  if return_type == 'dict':
    area_list = areas.getInfo().get('groups', [])

    area_labels = [d['classification'] for d in area_list]
    area_values = [d['sum'] for d in area_list]

    zonal_dict = {label: area for label, area in zip(area_labels, area_values)}
    return zonal_dict

  elif return_type == 'feature':
    area_list = ee.List(ee.Dictionary(areas).get('groups', ee.List([])))
    keys_list = area_list.map(
        lambda x: ee.Number(ee.Dictionary(x).get('classification')).format()
    )
    values_list = area_list.map(lambda x: ee.Dictionary(x).get('sum'))

    dict0 = ee.Dictionary.fromLists(keys_list, values_list)

    # Giữ lại thuộc tính thời gian để sort / debug nếu cần
    return ee.Feature(None, dict0).set(temporal_interval, ee.Image(classified).get(temporal_interval))

In [32]:
# === Olofsson et al. (2014) Area Estimation ===

def computeMapClassProportions(classified, roi, scale=None,
                               best_effort=False, max_pixels=1e13, tile_scale=16):
  """Compute Wi (area proportion of each map class) using pixel counting."""

  if isinstance(roi, ee.featurecollection.FeatureCollection):
    roi = roi.geometry()

  if scale is None:
    scale = resolve_analysis_scale(classified, fallback=30)

  areaImage = ee.Image.pixelArea().divide(1e6).addBands(classified)

  areas = areaImage.reduceRegion(
      reducer=ee.Reducer.sum().group(groupField=1, groupName='classification'),
      geometry=roi,
      scale=scale,
      maxPixels=max_pixels,
      bestEffort=best_effort,
      tileScale=tile_scale
  )

  area_list = areas.getInfo().get('groups', [])
  area_dict = {int(d['classification']): d['sum'] for d in area_list}
  total_area = sum(area_dict.values())

  Wi = {cls: area / total_area for cls, area in area_dict.items()} if total_area > 0 else {}

  return Wi, total_area, area_dict


def buildAreaProportionedMatrix(cm, Wi, class_labels):
  """Convert count-based confusion matrix to area-proportioned error matrix.
  Olofsson et al. (2014) equations 1-4.

  Args:
    cm: numpy array, standard confusion matrix (rows=map, cols=reference)
    Wi: dict, area proportions per class
    class_labels: list of class indices
  Returns:
    pandas DataFrame, area-proportioned error matrix
  """
  n_classes = len(class_labels)
  p_matrix = np.zeros((n_classes, n_classes))

  for i in range(n_classes):
    ni_dot = cm[i, :].sum()  # row total
    if ni_dot == 0:
      continue
    for j in range(n_classes):
      wi = Wi.get(class_labels[i], 0)
      p_matrix[i, j] = wi * cm[i, j] / ni_dot

  df = pd.DataFrame(p_matrix, index=CLASS_NAMES, columns=CLASS_NAMES)
  df.index.name = 'Map class'
  df.columns.name = 'Reference class'

  return df


def computeOlofssonMetrics(cm, Wi, totalArea, class_labels):
  """Compute Olofsson et al. (2014) adjusted accuracies, area estimates, and 95% CIs.

  Args:
    cm: numpy array, standard confusion matrix
    Wi: dict, area proportions
    totalArea: float, total area in km2
    class_labels: list of class indices
  Returns:
    pandas DataFrame with columns: Class, Map Area, Adjusted Area, CI, Adjusted UA, Adjusted PA
  """
  n_classes = len(class_labels)

  # Build area-proportioned matrix
  p_matrix = np.zeros((n_classes, n_classes))
  for i in range(n_classes):
    ni_dot = cm[i, :].sum()
    if ni_dot == 0:
      continue
    for j in range(n_classes):
      wi = Wi.get(class_labels[i], 0)
      p_matrix[i, j] = wi * cm[i, j] / ni_dot

  # Column sums = adjusted area proportions
  p_dot_j = p_matrix.sum(axis=0)

  # Adjusted User's Accuracy: p_ii / p_i.
  p_i_dot = p_matrix.sum(axis=1)
  adjusted_ua = np.zeros(n_classes)
  for i in range(n_classes):
    if p_i_dot[i] > 0:
      adjusted_ua[i] = p_matrix[i, i] / p_i_dot[i]

  # Adjusted Producer's Accuracy: p_ii / p_.j
  adjusted_pa = np.zeros(n_classes)
  for j in range(n_classes):
    if p_dot_j[j] > 0:
      adjusted_pa[j] = p_matrix[j, j] / p_dot_j[j]

  # Variance of area proportion (Olofsson eq. 10)
  var_p_j = np.zeros(n_classes)
  for j in range(n_classes):
    for i in range(n_classes):
      wi = Wi.get(class_labels[i], 0)
      ni_dot = cm[i, :].sum()
      if ni_dot <= 1:
        continue
      pij = cm[i, j] / ni_dot
      var_p_j[j] += (wi ** 2) * ((pij * (1 - pij)) / (ni_dot - 1))

  se_p_j = np.sqrt(var_p_j)

  # Area estimates and CIs
  adjusted_areas = p_dot_j * totalArea
  ci_areas = 1.96 * se_p_j * totalArea

  # Map areas
  map_areas = np.array([Wi.get(class_labels[i], 0) * totalArea for i in range(n_classes)])

  # Build results table
  results = pd.DataFrame({
      'Class': CLASS_NAMES,
      'Map Area (km2)': map_areas,
      'Adjusted Area (km2)': adjusted_areas,
      '95% CI (+/- km2)': ci_areas,
      'Adjusted UA': adjusted_ua,
      'Adjusted PA': adjusted_pa,
  })

  return results


In [33]:
# === Post-Classification Change Detection ===
import plotly.graph_objects as go

def _build_period_image(dates, roi):
  """Build a predictor image composite for a single time period.

  Args:
    dates: tuple (start_date, end_date) as strings
    roi: ee.Geometry or ee.FeatureCollection
  Returns:
    ee.Image with all predictor bands
  """
  collection = collections_list[satellite_list.index(w_satellite.value)]
  ind = satellite_list.index(w_satellite.value)
  bands_sel = list(w_select_bands.value)

  if w_harmonic_regression.value:
    imgCol = filterImageCollection(collection, roi, dates[0], dates[1], bands_sel, return_type='collection')
    if imgCol is None:
      return None
    imgCol = imgCol.map(lambda img: addSpectralIndices(img, ind, list(w_spectral_indices.value)))
    coefficients = fitHarmonicRegression(imgCol, 'EVI', w_harmonic_order.value)
    harmonicFeatures = extractHarmonicFeatures(coefficients, 'EVI', w_harmonic_order.value)
    if w_phenology_metrics.value:
      phenoMetrics = extractPhenologyMetrics(harmonicFeatures, 'EVI')
      img = harmonicFeatures.addBands(phenoMetrics)
    else:
      img = harmonicFeatures
  else:
    img = filterImageCollection(collection, roi, dates[0], dates[1], bands_sel)
    if img is None:
      return None
    img = addSpectralIndices(img, ind, list(w_spectral_indices.value))

  img = addVariables(img, list(w_variables.value), roi)

  if w_sar_fusion.value:
    s1 = getS1Composite(roi, dates[0], dates[1], w_speckle_radius.value)
    if s1 is not None:
      img = img.addBands(s1)

  # Remove constant band if present
  img_bands = img.bandNames().getInfo()
  if 'constant' in img_bands:
    img_bands.remove('constant')
    img = img.select(img_bands)

  return img

def _filter_labels_for_period(training_data, period_dates):
  """Filter dated labels to the requested period [start, end].

  If labels do not have a 'date' property, return the original collection.
  """
  if training_data is None:
    return None

  props = training_data.first().propertyNames()
  has_date = ee.List(props).contains('date')

  start_str = ee.String(period_dates[0])
  end_str = ee.String(period_dates[1])

  def _apply_filter():
    return training_data.filter(
        ee.Filter.And(
            ee.Filter.gte('date', start_str),
            ee.Filter.lte('date', end_str)
        )
    )

  return ee.FeatureCollection(
      ee.Algorithms.If(has_date, _apply_filter(), training_data)
  )


def _count_samples_per_class(fc, label_col):
  """Best-effort server-side class histogram for logging."""
  try:
    return ee.FeatureCollection(fc).aggregate_histogram(label_col).getInfo()
  except Exception:
    return None
def classifyTwoPeriods(period1_dates, period2_dates, roi, feature_names_list,
                       training_data=None, label_col=None,
                       retrain_per_period=True):
  """Classify LULC for two time periods using period-consistent training labels.

  Academic-safe logic:
  - Build one predictor composite per period.
  - If labels are dated, each period is trained only with labels whose
    dates fall inside that period.
  - If labels are not dated, fallback to the full label set.
  - Train one classifier per period to reduce temporal domain shift.
  """
  classified_images = []

  # Warn if periods overlap
  if period1_dates[1] >= period2_dates[0] and period2_dates[1] >= period1_dates[0]:
    print('Warning: Period 1 and Period 2 overlap in time. Interpret change results with caution.')

  for dates, label in [(period1_dates, 'Period 1'), (period2_dates, 'Period 2')]:
    print(f'  Processing {label}: {dates[0]} to {dates[1]}...')

    img = _build_period_image(dates, roi)
    if img is None:
      print(f'  Warning: No images for {label}')
      return None, None

    if retrain_per_period and training_data is not None and label_col is not None:
      # Filter labels to the current period when dated labels are available
      training_data_period = _filter_labels_for_period(training_data, dates)

      # Count filtered samples
      try:
        n_period = training_data_period.size().getInfo()
      except Exception:
        n_period = None

      if n_period is not None:
        print(f'  {label}: labels inside period = {n_period}')

      if n_period == 0:
        print(f'  Warning: No dated labels available inside {label}. Cannot train classifier for this period.')
        return None, None

      hist = _count_samples_per_class(training_data_period, label_col)
      if hist is not None:
        print(f'  {label}: class histogram = {hist}')

      # Re-sample only period-consistent labels on THIS period's predictor image
      period_samples = img.sampleRegions(
          collection=training_data_period,
          properties=[label_col],
          scale=resolve_analysis_scale(img),
          tileScale=4,
          geometries=False
      )

      # Remove rows with null predictors
      period_samples = period_samples.filter(
          ee.Filter.notNull(feature_names_list + [label_col])
      )

      try:
        n_period_samples = period_samples.size().getInfo()
      except Exception:
        n_period_samples = None

      if n_period_samples is not None:
        print(f'  {label}: usable sampled points = {n_period_samples}')

      if n_period_samples == 0:
        print(f'  Warning: No valid sampled predictors for {label} after filtering.')
        return None, None

      n_trees = int(classifier_param_dict.get('numberOfTrees', 100))
      period_clf = ee.Classifier.smileRandomForest(
          numberOfTrees=n_trees,
          seed=seed
      ).train(
          features=period_samples,
          classProperty=label_col,
          inputProperties=feature_names_list
      )

      classified = img.select(feature_names_list).classify(period_clf)
      print(f'  {label}: trained period-specific classifier ({n_trees} trees)')
    else:
      # Legacy fallback: use global classifier
      classified = img.select(feature_names_list).classify(classifier)
      print(f'  {label}: used global classifier')

    classified_images.append(classified)

  return classified_images[0], classified_images[1]


def computeTransitionMatrix(classified1, classified2, classNames, roi, scale=None):
  """Compute LULC transition matrix between two classified images.

  Returns:
  if isinstance(roi, ee.featurecollection.FeatureCollection):
    roi = roi.geometry()

  nClasses = len(classNames)

  # Encode transitions: from * N + to
  changeImage = classified1.multiply(nClasses).add(classified2)
  areaImage = ee.Image.pixelArea().divide(1e6).addBands(changeImage)

  areas = areaImage.reduceRegion(
      reducer=ee.Reducer.sum().group(groupField=1, groupName='transition'),
      geometry=roi,
      scale=scale,
      maxPixels=1e13,
      bestEffort=False,
      tileScale=4
  )

  area_list = areas.getInfo().get('groups', [])

  # Build transition matrix
  matrix = np.zeros((nClasses, nClasses))
  for item in area_list:
    code = int(item['transition'])
    from_cls = code // nClasses
    to_cls = code % nClasses
    if from_cls < nClasses and to_cls < nClasses:
      matrix[from_cls, to_cls] = item['sum']

  df = pd.DataFrame(matrix, index=classNames, columns=classNames)
  df.index.name = 'From (Period 1)'
    pandas DataFrame with from-class rows and to-class columns (area in km2)
  """
  df.columns.name = 'To (Period 2)'

  return df


def generateChangeMap(classified1, classified2, nClasses):
  """Generate a change map encoding transitions as from*N + to.
  Adds the layer to the geemap Map.
  """
  changeImage = classified1.multiply(nClasses).add(classified2)

  # Create a no-change mask
  noChange = classified1.eq(classified2)
  changeOnly = changeImage.updateMask(noChange.Not())

  change_palette = []
  for i in range(nClasses):
    for j in range(nClasses):
      change_palette.append(CLASS_PALETTE[i] if i < len(CLASS_PALETTE) else '#888888')

  Map.addLayer(changeOnly, {
      'min': 0,
      'max': nClasses * nClasses - 1,
      'palette': change_palette
  }, 'Change Map')

  return changeImage


def validateTransitions(transition_df, class_names, min_area_km2=0.5, flag_only=True):
  """Flag or filter implausible/small transitions.

  Adds columns to the transition DataFrame:
    - 'area_flag': 'below_threshold' if area < min_area_km2, else 'ok'
    - 'pct_of_change': percentage of total changed area

  Prints a summary of flagged transitions.

  Args:
    transition_df: pandas DataFrame (from computeTransitionMatrix / Fast)
    class_names: list of class names
    min_area_km2: float, minimum area threshold to flag noise
    flag_only: if True, only flag; if False, zero out flagged cells

  Returns:
    Validated DataFrame (copy)
  """
  df = transition_df.copy()
  n = len(class_names)

  # Compute total changed area (off-diagonal)
  total_change = 0.0
  for i in range(n):
    for j in range(n):
      if i != j:
        total_change += df.iloc[i, j]

  # Track flags
  n_small = 0
  small_transitions = []

  for i in range(n):
    for j in range(n):
      if i == j:
        continue
      area = df.iloc[i, j]
      if area > 0 and area < min_area_km2:
        n_small += 1
        pct = (area / total_change * 100) if total_change > 0 else 0
        small_transitions.append(
            f'  {class_names[i]} -> {class_names[j]}: {area:.3f} km2 ({pct:.2f}% of change)'
        )
        if not flag_only:
          df.iloc[i, j] = 0.0

  if n_small > 0:
    print(f'Transition validation: {n_small} transitions below {min_area_km2} km2 threshold')
    print('  (These may represent classification noise rather than real change)')
    for s in small_transitions[:10]:  # Show at most 10
      print(s)
    if n_small > 10:
      print(f'  ... and {n_small - 10} more')
  else:
    print(f'Transition validation: all off-diagonal transitions >= {min_area_km2} km2')

  return df


def renderSankeyDiagram(transitionDf, classNames, classPalette, minArea=1.0):
  """Render interactive Sankey diagram of LULC transitions using plotly.

  Args:
    transitionDf: pandas DataFrame (from computeTransitionMatrix)
    classNames: list of class names
    classPalette: list of hex color strings
    minArea: float, minimum area (km2) to include in diagram
  Returns:
    plotly Figure
  """
  nClasses = len(classNames)

  # Build Sankey data
  sources = []
  targets = []
  values = []
  link_colors = []

  for i in range(nClasses):
    for j in range(nClasses):
      area = transitionDf.iloc[i, j]
      if area >= minArea:
        sources.append(i)
        targets.append(nClasses + j)
        values.append(area)
        # Use source class color with transparency
        color = classPalette[i] if i < len(classPalette) else '#888888'
        hex_c = color.lstrip('#')
        r, g, b = int(hex_c[0:2], 16), int(hex_c[2:4], 16), int(hex_c[4:6], 16)
        link_colors.append(f'rgba({r},{g},{b},0.5)')


  # Node labels: Period 1 classes on left, Period 2 on right
  node_labels = [f'{name} (P1)' for name in classNames] + [f'{name} (P2)' for name in classNames]
  node_colors = classPalette[:nClasses] + classPalette[:nClasses]

  fig = go.Figure(data=[go.Sankey(
      node=dict(
          pad=15,
          thickness=20,
          line=dict(color='black', width=0.5),
          label=node_labels,
          color=node_colors
      ),
      link=dict(
          source=sources,
          target=targets,
          value=values,
          color=link_colors,
          hovertemplate='%{source.label} -> %{target.label}<br>Area: %{value:.1f} km\u00b2<extra></extra>'
      )
  )])

  fig.update_layout(
      title_text='LULC Transition Sankey Diagram',
      font_size=12,
      height=500
  )

  return fig


### Classify image collections

```
if 1:
  # either
  colMosaic_tmp_classified = getClassifiedCollections()
  df_tmpZonal = getClassifiedZonalAreas(classify=False)
else:
  # or
  df_tmpZonal = getClassifiedZonalAreas()

_ = interact(
  plotTemporalZonal,
  rot=w_rotation,
  select=w_zonal_selectClasses,
  chartType=w_zonal_chartType,
  subplot=w_zonal_subplot,
  df=fixed(df_tmpZonal)
)
```

In [34]:
def getClassifiedCollections():
  """Classify the temporal composite and calculate zonal areas

  Return classified image collection
  """
  global colMosaic_tmp_classified, listOfImages, col_dates_list

  def mapClassifyCollections(image):
      return ee.Image(image).select(bands).classify(classifier) \
              .copyProperties(image, [temporal_interval])

  if colMosaic_tmp == None:
    raise Exception("Temporal composite None for all-time interval.")

  colMosaic_tmp_classified = colMosaic_tmp.map(mapClassifyCollections)
  w_classified_tmpflag.value = True
  listOfImages = colMosaic_tmp_classified.toList(colMosaic_tmp_classified.size())

  col_dates = colMosaic_tmp.aggregate_array(temporal_interval).distinct().sort()
  col_dates_list = col_dates.getInfo()

  return colMosaic_tmp_classified


def getClassifiedZonalAreas(roi=None, classify=True):
  """Calculate zonal areas for the classified collection

  Args
  roi: region of interest, default to the entire region
       could be a subarea recognized as a hotspot area
  classify: flag to classify the image collection

  Return zonal feature collection
  """
  global colMosaic_tmp_classified, df_tmpZonal, col_dates_list

  if roi == None:
    roi = region

  if classify:
    colMosaic_tmp_classified = getClassifiedCollections()

  print('Calculating zonal areas for the classified composite ...')
  colMosaic_tmp_zonal = ee.FeatureCollection(
    colMosaic_tmp_classified.map(
        lambda classified: zonalStatistics(classified, roi, 'feature')
    )
  ).sort(temporal_interval)

  df = geemap.ee_to_df(colMosaic_tmp_zonal)

  col_dates = colMosaic_tmp.aggregate_array(temporal_interval).distinct().sort()
  col_dates_list = col_dates.getInfo()

  if temporal_interval == 'year':
    fmt = '%Y'
  elif temporal_interval == 'month':
    fmt = '%Y-%m'
  else:
    fmt = '%Y-%m-%d'

  # Tách cột thời gian ra nếu geemap đã đưa nó vào dataframe
  time_col = temporal_interval if temporal_interval in df.columns else None

  if time_col is not None:
    time_values = df[time_col]
    class_cols = [c for c in df.columns if c != time_col]
  else:
    time_values = None
    class_cols = list(df.columns)

  # Chỉ giữ và đổi tên các cột class
  df = df[class_cols]

  renamed_class_cols = []
  for c in class_cols:
    try:
      renamed_class_cols.append(CLASS_NAMES[int(c)])
    except Exception:
      renamed_class_cols.append(str(c))

  df.columns = renamed_class_cols

  # Dùng cột thời gian làm index nếu có; nếu không thì fallback về col_dates_list
  if time_values is not None:
    df.index = pd.to_datetime(time_values.astype(str), format=fmt)
  else:
    df.index = pd.to_datetime(col_dates_list, format=fmt)

  df_tmpZonal = df
  return df_tmpZonal

###  tabs

* Labels histogram
* Parallel coordinate plot
* Classifier assessment
* Feature importance
* Shap values
* Zonal statistics
* Spatio-temporal dynamics

#### Plot functions

In [35]:
import shap
from ipywidgets import interact, interactive, interactive_output, fixed
import seaborn as sns
import plotly.express as px

def plotLabelsHist(width, rot, label_hist):
  def _safe_class_name(k):
    try:
      return CLASS_NAMES[int(k)]
    except (ValueError, TypeError):
      return k
  def _safe_class_color(k):
    try:
      return CLASS_PALETTE[int(k)]
    except (ValueError, TypeError):
      idx = CLASS_NAMES.index(k) if k in CLASS_NAMES else 0
      return CLASS_PALETTE[idx]
  x = [_safe_class_name(k) for k in label_hist.keys()]
  y = list(label_hist.values())
  pal = [_safe_class_color(k) for k in label_hist.keys()]

  fig1, ax1 = plt.subplots()
  sns.barplot(x=x, y=y, ax=ax1, palette=pal)
  plt.xticks(rotation=rot)
  ax1.set_ylabel('Counts')
  change_barplot_width(ax1, width)
  ax1.set_title('Labels class distribution')
  plt.show()

def plotZonalAreas(width, rot, zonal_dict):
  #zonal_dict = zonalStatistics(image_classified)
  #print(zonal_dict)

  def _safe_class_name(k):
    try:
      return CLASS_NAMES[int(k)]
    except (ValueError, TypeError):
      return k
  def _safe_class_color(k):
    try:
      return CLASS_PALETTE[int(k)]
    except (ValueError, TypeError):
      idx = CLASS_NAMES.index(k) if k in CLASS_NAMES else 0
      return CLASS_PALETTE[idx]
  fig, ax2 = plt.subplots()
  x = [_safe_class_name(k) for k in zonal_dict.keys()]
  pal = [_safe_class_color(k) for k in zonal_dict.keys()]
  area_values = list(zonal_dict.values())

  sns.barplot(x=x, y=area_values, ax=ax2, palette=pal)
  plt.xticks(rotation=rot)
  change_barplot_width(ax2, width)
  ax2.set_ylabel('$Areas\ (km^{2})$')
  ax2.set_title('Class area distribution for predicted image')
  plt.show()

  # Tabulate the zonal areas and the percentage
  zonalAreas = pd.DataFrame({'Areas (km^2)': area_values}, index=x)
  zonalAreas['Percentage (%)'] = zonalAreas['Areas (km^2)']/sum(area_values)*100
  zonalAreas.loc["Sum of classes"] = zonalAreas.sum()
  print('\n', zonalAreas, '\n')

def plotConfusionMatrix(normalize, gee_ytest, gee_ypred, y_test, y_pred):
  fig, (ax1, ax2) = plt.subplots(1,2, figsize=(14,6))
  labels = list(range(0, len(CLASS_NAMES)))
  if normalize == '-':
    normalize = None
  cm1 = confusion_matrix(gee_ytest, gee_ypred, labels=labels, normalize=normalize)
  cm2 = confusion_matrix(y_test, y_pred, labels=labels, normalize=normalize)

  sns.heatmap(cm1, annot=True, ax=ax1, cmap='Blues',
              xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
  ax1.set(xlabel="Predicted", ylabel="True", title='GEE')

  # cmap='Blues' or "OrRd"
  sns.heatmap(cm2, annot=True, ax=ax2, cmap='Blues',
              xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
  ax2.set(xlabel="Predicted", ylabel="True", title='sklearn')
  plt.show()


In [36]:
def plotTemporalZonal(rot, select, chartType, subplot, df):

    # Guard: check temporal_interval is defined before use
    if 'temporal_interval' not in globals() or temporal_interval is None:
        print('temporal_interval is not set. Please run classification first.')
        return

    # ===============================
    cols_to_plot = [c for c in select if c in df.columns]

    if len(cols_to_plot) == 0:
        print("⚠️ Không có dữ liệu zonal cho class đã chọn")
        return

    # lấy màu theo class name
    colors = [CLASS_PALETTE[CLASS_NAMES.index(c)] for c in cols_to_plot]

    figsize = (11, 6)

    # ===============================
    # LINE CHART
    # ===============================
    if chartType == 'line':

        if not subplot or (subplot and len(cols_to_plot) == 1):
            fig, ax = plt.subplots(figsize=figsize)
            df.loc[:, cols_to_plot].plot(ax=ax, color=colors)
            ax.set_ylabel('$Areas\ (km^{2})$')

        else:
            fig, ax = plt.subplots(len(cols_to_plot), 1, figsize=figsize)

            for ind, col in enumerate(cols_to_plot):
                df[col].plot(ax=ax[ind],
                             color=CLASS_PALETTE[CLASS_NAMES.index(col)])
                ax[ind].set_ylabel('$Areas\ (km^{2})$')

                if ind < len(cols_to_plot) - 1:
                    ax[ind].set_xticklabels('')
                    ax[ind].set_xticklabels('', minor=True)

        fig.autofmt_xdate(rotation=rot, ha='center')
        plt.show()

    # ===============================
    # BAR CHART
    # ===============================
    elif chartType == 'bar':

        # xử lý tick label thời gian
        if temporal_interval == 'year':
            tick_labels = [x.strftime("%Y") for x in df.index]
        else:
            tick_labels = [
                x.strftime("%m\n%Y") if x.month == 1 else x.strftime('%m')
                for x in df.index
            ]
            tick_labels = [tick.lstrip("0") for tick in tick_labels]

        nplots = len(cols_to_plot)

        if subplot and nplots > 1:
            ax = df.loc[:, cols_to_plot].plot.bar(
                figsize=figsize,
                subplots=True,
                color=colors,
                rot=rot
            )

            for i in range(nplots):
                ax[i].set_title('')
                ax[i].set_ylabel('$Areas\ (km^{2})$')

            ax[nplots - 1].set_xticklabels(tick_labels)

        else:
            fig, ax = plt.subplots(figsize=figsize)
            df.loc[:, cols_to_plot].plot.bar(
                ax=ax,
                color=colors,
                rot=rot
            )
            ax.set_xticklabels(tick_labels)
            ax.set_ylabel('$Areas\ (km^{2})$')

        plt.show()


In [37]:
def plotClassifiedGrid(select_dates, grid_column, figsize, dim):
  '''Plot image thumbnails in a grid for selected dates
  First export the thumbnail, then imshow
  '''
  global listOfImages, colMosaic_tmp_classified, col_dates_list

  colMosaic_tmp_classified = getClassifiedCollections()

  if listOfImages is None:
    listOfImages = colMosaic_tmp_classified.toList(colMosaic_tmp_classified.size())

  if not select_dates:
        print("Vui lòng chọn ít nhất 1 ngày.")
        return

  cols = grid_column
  img_count = len(select_dates)
  rows = int(np.ceil(img_count / cols))
  cols = grid_column

  img_count = len(select_dates)
  rows = int(np.ceil(img_count / cols))

  figsize = [int(f) for f in figsize.split(',')]

  print(img_count, 'dates selected')

  col_index = [col_dates_list.index(d) for d in select_dates]

  resultVis1 = {
      'min': 0,
      'max': len(CLASS_NAMES) - 1,
      'palette': CLASS_PALETTE
  }

  if isinstance(region, ee.featurecollection.FeatureCollection):
    roi = region.geometry()
  else:
    roi = region

  imgs = []

  for s in col_index:

    out_img = 'tmp_thumbnail_{}.png'.format(col_dates_list[s])

    if os.path.isfile(out_img):
      print('Found existing', out_img)
      imgs.append(plt.imread(out_img))

    else:
      print('Exporting {} ...'.format(out_img))

      result = ee.Image(listOfImages.get(s))

      geemap.get_image_thumbnail(
          result,
          out_img,
          resultVis1,
          dimensions=dim,
          region=roi
      )

      imgs.append(plt.imread(out_img))

  print('\n')

  fig, axes = plt.subplots(
      nrows=rows,
      ncols=cols,
      figsize=figsize
  )

  # đảm bảo axes luôn là array
  axes = np.array(axes).reshape(-1)

  for i, ax in enumerate(axes):

    if i >= img_count:
      ax.set_axis_off()
      continue

    ax.imshow(imgs[i])
    ax.axis('off')
    ax.set_title(col_dates_list[col_index[i]])

  plt.show()


In [38]:
# some barplot utility functions
def change_barplot_width(ax, new_value):
  for patch in ax.patches :
    current_width = patch.get_width()
    diff = current_width - new_value

    # we change the bar width
    patch.set_width(new_value)

    # we recenter the bar
    patch.set_x(patch.get_x() + diff * .5)

def change_barhplot_width(ax, new_value):
  for patch in ax.patches :
    current_height = patch.get_height()
    diff = current_height - new_value

    # we change the bar width
    patch.set_height(new_value)

    # we recenter the bar
    patch.set_y(patch.get_y() + diff * .5)

In [39]:
def _create_tabs(tab_titles):
    tab = widgets.Tab()
    tab.children = [widgets.Output() for _ in tab_titles]

    for i, title in enumerate(tab_titles):
        tab.set_title(i, title)

    return tab


In [40]:
# ============================================================
# Analysis cache + lazy/on-demand helpers
# Place this block before createTabs()
# ============================================================

analysis_cache = {
    'olofsson': None,
    'change_detection': None,
}

def clear_analysis_cache(*keys):
    global analysis_cache
    if not keys:
        analysis_cache = {
            'olofsson': None,
            'change_detection': None,
        }
    else:
        for key in keys:
            analysis_cache[key] = None

def _remove_map_layer_by_name(layer_name):
    for layer in list(Map.layers):
        if getattr(layer, 'name', None) == layer_name:
            Map.remove(layer)

def _concat_cv_predictions(cv_results):
    y_true = []
    y_pred = []
    for res in cv_results:
        # Support both snake_case (y_true/y_pred) and lowercase (ytrue/ypred)
        if 'ytrue' in res and 'ypred' in res:
            y_true.append(np.asarray(res['ytrue']))
            y_pred.append(np.asarray(res['ypred']))
        elif 'y_true' in res and 'y_pred' in res:
            y_true.append(np.asarray(res['y_true']))
            y_pred.append(np.asarray(res['y_pred']))
        else:
            raise ValueError('Spatial CV results missing ytrue/y_true or ypred/y_pred key.')
    if not y_true or not y_pred:
        raise ValueError('Spatial CV results do not contain ytrue/ypred.')
    return np.concatenate(y_true), np.concatenate(y_pred)

def getEvaluationArrays(spatial_mode, holdout_mode, cv_fold_results=None, y_test=None, y_pred=None):
    labels = list(range(len(CLASS_NAMES)))

    if spatial_mode and cv_fold_results is not None:
        y_true_eval, y_pred_eval = _concat_cv_predictions(cv_fold_results)
    elif holdout_mode and y_test is not None and y_pred is not None:
        y_true_eval = np.asarray(y_test)
        y_pred_eval = np.asarray(y_pred)
    else:
        raise ValueError('No valid predictions available for assessment.')

    cm = confusion_matrix(y_true_eval, y_pred_eval, labels=labels)
    return labels, y_true_eval, y_pred_eval, cm

def computeMapClassProportionsFast(classified, roi, scale=None,
                                   tile_scale=16, max_pixels=1e13, best_effort=False):
    if isinstance(roi, ee.featurecollection.FeatureCollection):
        roi = roi.geometry()

    if scale is None:
        scale = resolve_analysis_scale(classified, fallback=30)

    area_image = ee.Image.pixelArea().divide(1e6).addBands(classified)

    areas = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='classification'),
        geometry=roi,
        scale=scale,
        maxPixels=max_pixels,
        bestEffort=best_effort,
        tileScale=tile_scale
    )

    area_list = areas.getInfo().get('groups', [])
    area_dict = {int(d['classification']): d['sum'] for d in area_list}
    total_area = sum(area_dict.values())
    Wi = {cls: area / total_area for cls, area in area_dict.items()} if total_area > 0 else {}

    return Wi, total_area, area_dict

def computeTransitionMatrixFast(classified1, classified2, class_names, roi, scale=None, tile_scale=16, max_pixels=1e13, best_effort=False):
    if isinstance(roi, ee.featurecollection.FeatureCollection):
        roi = roi.geometry()
        if scale is None:
            scale = resolve_analysis_scale(classified1, fallback=30)
    n_classes = len(class_names)
    change_image = classified1.multiply(n_classes).add(classified2)
    area_image = ee.Image.pixelArea().divide(1e6).addBands(change_image)

    areas = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='transition'),
        geometry=roi,
        scale=scale,
        maxPixels=max_pixels,
        bestEffort=best_effort,
        tileScale=tile_scale
    )

    groups = (areas.getInfo() or {}).get('groups', [])
    matrix = np.zeros((n_classes, n_classes), dtype=float)

    for item in groups:
        code = int(item['transition'])
        from_cls = code // n_classes
        to_cls = code % n_classes
        if from_cls < n_classes and to_cls < n_classes:
            matrix[from_cls, to_cls] = float(item['sum'])

    df = pd.DataFrame(matrix, index=class_names, columns=class_names)
    df.index.name = 'From Period 1'
    df.columns.name = 'To Period 2'
    return df


#### Feature importance function

Not just feature importance, actually.

In [41]:
def createTabs():
    """
    Academic-safe version:
    - Spatial Block CV: evaluate only with out-of-fold predictions.
    - Holdout mode: compare GEE vs sklearn on a true test split.
    - Holdout-only tabs (parallel plot, feature importance, SHAP) are disabled in spatial CV mode.
    """

    global forest, shap_values, shap_obj
    global X_train, X_test, y_train, y_test

    if not w_classified_flag.value:
        print('Classify the image first.')
        return

    if w_classifier_select.value != 'RandomForest':
        print('Select RandomForest to display feature importance.')
        return

    spatial_mode = bool(w_spatial_block_cv.value and cv_fold_results is not None)
    holdout_mode = bool((not w_spatial_block_cv.value) and testing is not None)

    forest = None
    shap_values = None
    shap_obj = None
    X_train = X_test = y_train = y_test = None

    gee_y_test = gee_y_pred = None
    y_pred = None

    def cv_concat(results, key1, key2=None):
        arrs = []
        for res in results:
            if key1 in res:
                arrs.append(np.asarray(res[key1]))
            elif key2 is not None and key2 in res:
                arrs.append(np.asarray(res[key2]))
            else:
                raise KeyError(f"Missing keys {key1}/{key2} in cv_fold_results.")
        return np.concatenate(arrs)

    def safe_feature_idx(idx, fallback=0):
        if len(feature_names) == 0:
            return None
        if idx < len(feature_names):
            return feature_names[idx]
        return feature_names[fallback]

    tab_names = ['Labels histogram', 'Classifier assessment', 'Zonal statistics']

    if holdout_mode:
        tab_names.insert(1, 'Parallel coordinate plot')
        tab_names.insert(3, 'Feature importance')
        tab_names.insert(4, 'SHAP values')

    if temporal_interval != 'all-time':
        tab_names.append('Spatio-temporal dynamics')

    tab_names.append('Olofsson Assessment')

    if w_change_detection.value:
        tab_names.append('Change Detection')

    ctab = _create_tabs(tab_names)
    display(ctab)

    tab_idx = {name: i for i, name in enumerate(tab_names)}

    color_bar_plot = 'blue'

    w_barWidth = widgets.FloatSlider(
        description='Bar width',
        value=0.5,
        min=0.05,
        max=1,
        step=0.05,
        continuous_update=False,
        style=desc_width
    )

    w_rotation = widgets.IntSlider(
        value=45,
        min=0,
        max=90,
        step=1,
        description='Label rotation',
        continuous_update=False,
        style=desc_width
    )

    cm_normalize_options = [
        ('normalized over the true conditions', 'true'),
        ('normalized over the predicted conditions', 'pred'),
        ('normalized by the total number of samples', 'all'),
        ('not normalized', '-')
    ]

    w_cm_normalize = widgets.Dropdown(
        options=cm_normalize_options,
        value='-',
        description='Confusion matrix',
        layout=widgets.Layout(width='max-content'),
        style=desc_width
    )

    if holdout_mode:
        print('Run scikit-learn classifier ...')
        forest, X_train, X_test, y_train, y_test = sklearnClassifier(training, testing)
        y_pred = forest.predict(X_test)

        test_classified = testing.classify(classifier)
        gee_result = geemap.ee_to_df(test_classified.select([LABEL, 'classification']))
        gee_y_test = gee_result[LABEL]
        gee_y_pred = gee_result['classification']

        print('Calculate feature importance ...')
        rf_explain = classifier.explain().getInfo()
        gee_importance = pd.Series(rf_explain['importance']).sort_values(ascending=True)

        mdi_std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)
        mdi_importances = pd.Series(forest.feature_importances_, index=feature_names).sort_values(ascending=True)

        permu_test_result = permutation_importance(
            forest, X_test, y_test,
            n_repeats=30,
            random_state=seed,
            n_jobs=5
        )
        permu_test_importances = pd.Series(
            permu_test_result.importances_mean,
            index=feature_names
        ).sort_values(ascending=True)

        sorted_test_idx = permu_test_result.importances_mean.argsort()
        permu_test_df = pd.DataFrame(
            permu_test_result.importances[sorted_test_idx].T,
            columns=[feature_names[i] for i in sorted_test_idx]
        )

        print('Calculate SHAP values ...')
        explainer = shap.TreeExplainer(forest)
        shap_values = explainer.shap_values(X_test)
        shap_obj = explainer(X_test)

        test_hist = testing.aggregate_histogram(LABEL).getInfo()
        if len(test_hist.keys()) != len(CLASS_NAMES):
            CLASS_CODES_shap = [int(k) for k in test_hist.keys()]
            CLASS_NAMES_shap = [CLASS_NAMES[i] for i in CLASS_CODES_shap]
            CLASS_PALETTE_shap = [CLASS_PALETTE[i] for i in CLASS_CODES_shap]
        else:
            CLASS_CODES_shap = list(range(len(CLASS_NAMES)))
            CLASS_NAMES_shap = CLASS_NAMES
            CLASS_PALETTE_shap = CLASS_PALETTE

        w_shap_class = widgets.Dropdown(
            options=CLASS_NAMES_shap,
            value=CLASS_NAMES_shap[0],
            description='Select class name',
            layout=widgets.Layout(width='max-content'),
            style=desc_width
        )

        w_shap_feature0 = widgets.Dropdown(
            options=feature_names,
            value=safe_feature_idx(0),
            description='Main feature',
            layout=widgets.Layout(width='max-content'),
            style=desc_width
        )

        w_shap_feature1 = widgets.Dropdown(
            options=feature_names,
            value=safe_feature_idx(1),
            description='Interaction feature',
            layout=widgets.Layout(width='max-content'),
            style=desc_width
        )

    w_zonal_selectClasses = widgets.SelectMultiple(
        options=CLASS_NAMES,
        value=CLASS_NAMES[:min(3, len(CLASS_NAMES))],
        description='Select classes',
        rows=6,
        disabled=False,
        style=desc_width
    )

    w_zonal_chartType = widgets.Dropdown(
        options=['line', 'bar'],
        value='line',
        description='Chart type',
        layout=widgets.Layout(width='max-content'),
        style=desc_width
    )

    w_zonal_subplot = widgets.Checkbox(
        description='Subplot',
        value=False,
        style=desc_width
    )

    # -------------------------
    # TAB: Labels histogram
    # -------------------------
    with ctab.children[tab_idx['Labels histogram']]:
        ctab.children[tab_idx['Labels histogram']].clear_output()

        if holdout_mode:
            train_hist = training.aggregate_histogram(LABEL).getInfo()
            test_hist2 = testing.aggregate_histogram(LABEL).getInfo()

            train_vals, test_vals, all_vals = [], [], []

            for i in range(len(CLASS_NAMES)):
                key = str(i)
                tr_val = train_hist.get(key, train_hist.get(i, 0))
                te_val = test_hist2.get(key, test_hist2.get(i, 0))
                train_vals.append(tr_val)
                test_vals.append(te_val)
                all_vals.append(tr_val + te_val)

            df = pd.DataFrame(
                {
                    'All labels used': all_vals,
                    'Train labels': train_vals,
                    'Test labels': test_vals,
                },
                index=CLASS_NAMES
            )
            df.loc['Sum of classes'] = df.sum()

            label_hist_used = dict(zip([str(i) for i in range(len(CLASS_NAMES))], all_vals))
            interact(
                plotLabelsHist,
                width=w_barWidth,
                rot=w_rotation,
                label_hist=fixed(label_hist_used)
            )
            print(df)

        else:
            all_hist = training.aggregate_histogram(LABEL).getInfo()

            all_vals = []
            for i in range(len(CLASS_NAMES)):
                key = str(i)
                all_vals.append(all_hist.get(key, all_hist.get(i, 0)))

            df = pd.DataFrame(
                {'All labels used': all_vals},
                index=CLASS_NAMES
            )
            df.loc['Sum of classes'] = df.sum()

            label_hist_used = dict(zip([str(i) for i in range(len(CLASS_NAMES))], all_vals))
            interact(
                plotLabelsHist,
                width=w_barWidth,
                rot=w_rotation,
                label_hist=fixed(label_hist_used)
            )
            print(df)
            print('Block CV mode: no single Train/Test split is reported.')

    # --------------------------------
    # TAB: Parallel coordinate plot
    # --------------------------------
    if holdout_mode:
        with ctab.children[tab_idx['Parallel coordinate plot']]:
            ctab.children[tab_idx['Parallel coordinate plot']].clear_output()

            print('Parallel coordinate plot for test data')
            test_data = geemap.ee_to_df(testing)

            n_classes = len(CLASS_NAMES)
            color_scale = []
            for i in range(n_classes):
                color_scale.append([float(i / n_classes), CLASS_PALETTE[i]])
                color_scale.append([float((i + 1) / n_classes), CLASS_PALETTE[i]])
            n_dims = len(feature_names) + 1  # +1 cho cột label
            dynamic_width = max(800, n_dims * 80)
            fig = px.parallel_coordinates(
                test_data,
                color=LABEL,
                dimensions=feature_names + [LABEL],
                color_continuous_scale=color_scale,
            )
            fig.update_layout(
                width=dynamic_width,
                height=500,
                margin=dict(l=60, r=60, t=60, b=60),
                font=dict(size=11),
                coloraxis_colorbar=dict(
                    title=LABEL,
                    tickvals=list(range(n_classes)),
                    ticktext=CLASS_NAMES,
                )
            )
            fig.show()

    # ---------------------------
    # TAB: Classifier assessment
    # ---------------------------
    with ctab.children[tab_idx['Classifier assessment']]:
        ctab.children[tab_idx['Classifier assessment']].clear_output()

        if spatial_mode:
            display(widgets.HTML("<h3 style='text-align:center'>Spatial Block Cross-Validation Metrics (Out-of-fold)</h3>"))
            print('Đây là độ chính xác ngoài mẫu theo không gian, phù hợp hơn cho đánh giá học thuật.')
            print('NOTE: CV metrics below are computed from OUT-OF-FOLD predictions (spatially independent).')
            print('  The final classified map uses a model trained on ALL samples.')
            print('  This is standard practice (Wainer & Cawley 2021) but means map accuracy')
            print('  may slightly differ from reported CV metrics.')

            y_true_cv = cv_concat(cv_fold_results, 'ytrue', 'y_true')
            y_pred_cv = cv_concat(cv_fold_results, 'ypred', 'y_pred')
            _cv_summary = aggregateCVResults(cv_fold_results, verbose=False)

            classifierMetrics(y_true_cv, y_pred_cv)

            fig, ax = plt.subplots(figsize=(7, 6))
            labels = list(range(len(CLASS_NAMES)))
            norm = None if w_cm_normalize.value == '-' else w_cm_normalize.value
            cm_cv = confusion_matrix(y_true_cv, y_pred_cv, labels=labels, normalize=norm)
            sns.heatmap(cm_cv, annot=True, ax=ax, cmap='Blues',
                        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
            ax.set(xlabel='Predicted', ylabel='True', title='Spatial Block CV Confusion Matrix')
            plt.show()

            # Statistical tests for spatial CV
            from scipy import stats as _stats
            _oa_cv = accuracy_score(y_true_cv, y_pred_cv)
            _kappa_cv = cohen_kappa_score(y_true_cv, y_pred_cv)
            _n_cv = len(y_true_cv)
            _se0_cv = 1.0 / np.sqrt(_n_cv)
            _z_cv = _kappa_cv / _se0_cv
            _p_cv = 2 * (1 - _stats.norm.cdf(abs(_z_cv)))
            print(f'\nOA = {_oa_cv:.4f}, Kappa = {_kappa_cv:.4f}, QD = {_cv_summary.get("mean_qd", float("nan")):.4f}, AD = {_cv_summary.get("mean_ad", float("nan")):.4f}')
            print(f'Kappa significance: z = {_z_cv:.4f}, p = {_p_cv:.6f}')
            if _p_cv < 0.05:
                print('  -> Kappa is significantly different from 0 (p < 0.05)')

        elif holdout_mode:
            left = widgets.Output()
            right = widgets.Output()
            display(widgets.HBox([left, right]))

            with left:
                display(widgets.HTML("<h3 style='text-align:center'>GEE metrics (Test set)</h3>"))
                classifierMetrics(gee_y_test, gee_y_pred)

            with right:
                display(widgets.HTML("<h3 style='text-align:center'>Scikit-learn metrics (Test set)</h3>"))
                classifierMetrics(y_test, y_pred)

            print()
            interact(
                plotConfusionMatrix,
                normalize=w_cm_normalize,
                gee_ytest=fixed(gee_y_test),
                gee_ypred=fixed(gee_y_pred),
                y_test=fixed(y_test),
                y_pred=fixed(y_pred)
            )

            # ── Statistical Significance Tests ──────────────────────
            from scipy import stats as _stats

            def _bootstrap_accuracy_ci(y_true, y_pred_arr, n_bootstrap=1000, ci=0.95, seed_val=42):
                rng = np.random.RandomState(seed_val)
                n = len(y_true)
                accs = [accuracy_score(y_true[idx], y_pred_arr[idx])
                        for idx in (rng.randint(0, n, n) for _ in range(n_bootstrap))]
                lo = np.percentile(accs, (1 - ci) / 2 * 100)
                hi = np.percentile(accs, (1 + ci) / 2 * 100)
                return np.mean(accs), lo, hi

            def _kappa_significance(y_true, y_pred_arr):
                kappa = cohen_kappa_score(y_true, y_pred_arr)
                n = len(y_true)
                se0 = 1.0 / np.sqrt(n)  # Fleiss (1971) SE under H0
                z = kappa / se0
                p = 2 * (1 - _stats.norm.cdf(abs(z)))
                return kappa, se0, z, p

            print('\n' + '=' * 60)
            print('STATISTICAL SIGNIFICANCE TESTS')
            print('=' * 60)

            # Bootstrap CI for sklearn RF
            y_test_arr = np.asarray(y_test)
            y_pred_arr = np.asarray(y_pred)
            mean_acc, ci_lo, ci_hi = _bootstrap_accuracy_ci(y_test_arr, y_pred_arr)
            print(f'\nsklearn RF  OA = {mean_acc:.4f}  (95% Bootstrap CI: [{ci_lo:.4f}, {ci_hi:.4f}])')

            # Quantity & Allocation Disagreement – sklearn RF (Pontius & Millones 2011)
            _cm_sk = confusion_matrix(y_test_arr, y_pred_arr)
            _cm_prop_sk = _cm_sk.astype(float) / _cm_sk.sum()
            _qd_sk = np.sum(np.abs(_cm_prop_sk.sum(axis=1) - _cm_prop_sk.sum(axis=0))) / 2
            _ad_sk = (1 - mean_acc) - _qd_sk

            kappa, se0, z, p_k = _kappa_significance(y_test_arr, y_pred_arr)
            print(f'sklearn RF  Kappa = {kappa:.4f}, SE0 = {se0:.4f}, z = {z:.4f}, p = {p_k:.6f}  |  QD = {_qd_sk:.4f}, AD = {_ad_sk:.4f}')
            if p_k < 0.05:
                print('  -> Kappa is significantly different from 0 (p < 0.05)')
            print(f'sklearn RF  Quantity disagreement = {_qd_sk:.4f}, Allocation disagreement = {_ad_sk:.4f}')

            # Bootstrap CI for GEE RF
            gee_yt = np.asarray(gee_y_test)
            gee_yp = np.asarray(gee_y_pred)
            gee_acc, gee_lo, gee_hi = _bootstrap_accuracy_ci(gee_yt, gee_yp)
            print(f'\nGEE RF      OA = {gee_acc:.4f}  (95% Bootstrap CI: [{gee_lo:.4f}, {gee_hi:.4f}])')

            # Quantity & Allocation Disagreement – GEE RF (Pontius & Millones 2011)
            _cm_gee = confusion_matrix(gee_yt, gee_yp)
            _cm_prop_gee = _cm_gee.astype(float) / _cm_gee.sum()
            _qd_gee = np.sum(np.abs(_cm_prop_gee.sum(axis=1) - _cm_prop_gee.sum(axis=0))) / 2
            _ad_gee = (1 - gee_acc) - _qd_gee

            gee_kappa, gee_se0, gee_z, gee_pk = _kappa_significance(gee_yt, gee_yp)
            print(f'GEE RF      Kappa = {gee_kappa:.4f}, SE0 = {gee_se0:.4f}, z = {gee_z:.4f}, p = {gee_pk:.6f}  |  QD = {_qd_gee:.4f}, AD = {_ad_gee:.4f}')
            print(f'GEE RF      Quantity disagreement = {_qd_gee:.4f}, Allocation disagreement = {_ad_gee:.4f}')

            # McNemar test: GEE vs sklearn
            correct_gee = (gee_yp == gee_yt)
            correct_sk  = (y_pred_arr == y_test_arr)
            b = int(np.sum(correct_gee & ~correct_sk))
            c = int(np.sum(~correct_gee & correct_sk))

            print(f"\nMcNemar's test (GEE-RF vs sklearn-RF):")
            print(f'  Discordant pairs: b={b} (GEE correct, sklearn wrong), c={c} (sklearn correct, GEE wrong)')
            if b + c > 0:
                if b + c < 25:
                    p_mcn = _stats.binomtest(min(b, c), b + c, 0.5).pvalue
                else:
                    chi2 = (abs(b - c) - 1) ** 2 / (b + c)
                    p_mcn = 1 - _stats.chi2.cdf(chi2, df=1)
                print(f'  p-value = {p_mcn:.6f}')
                if p_mcn < 0.05:
                    print('  -> Classifiers are significantly different (p < 0.05)')
                else:
                    print('  -> No significant difference between classifiers (p >= 0.05)')
            else:
                print('  No discordant pairs -- classifiers agree on all samples.')

        else:
            print('No valid evaluation mode found.')

    # ------------------------
    # TAB: Feature importance
    # ------------------------
    if holdout_mode:
        with ctab.children[tab_idx['Feature importance']]:
            ctab.children[tab_idx['Feature importance']].clear_output()

            out0 = widgets.Output()
            out1 = widgets.Output()
            out2 = widgets.Output()
            display(widgets.VBox([out0, out1, out2]))

            with out0:
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

                gee_importance.plot.barh(ax=ax1, color=color_bar_plot)
                ax1.set_title('GEE')
                ax1.set_xlabel('Decrease in impurity (sum)')

                mdi_importances.plot.barh(ax=ax2, color=color_bar_plot)
                ax2.set_title('sklearn')
                ax2.set_xlabel('Mean decrease in impurity (normalized)')

                plt.suptitle('Impurity-based feature importance', fontsize=14)
                plt.show()

            with out1:
                fig, ax = plt.subplots(1, 1, figsize=(8, 6))

                permu_test_importances.plot.barh(ax=ax, color=color_bar_plot)
                ax.set_title('Permutation Importance (Test Set)')
                ax.set_xlabel('Mean decrease in accuracy')

                plt.suptitle('Permutation feature importance (barplot)', fontsize=14)
                plt.tight_layout()
                plt.show()

            with out2:
                fig, ax = plt.subplots(1, 1, figsize=(8, 6))

                permu_test_df.boxplot(ax=ax, grid=False, vert=False)
                ax.set_title('Permutation Importance (Test Set, 30 repeats)')
                ax.set_xlabel('Mean decrease in accuracy')

                plt.suptitle('Permutation feature importance (boxplot)', fontsize=14)
                plt.tight_layout()
                plt.show()

    # ------------------------
    # TAB: SHAP values
    # ------------------------
    if holdout_mode:
        with ctab.children[tab_idx['SHAP values']]:
            ctab.children[tab_idx['SHAP values']].clear_output()

            import numpy as _np

            def _get_class_index(shap_class_name: str) -> int:
                return CLASS_NAMES_shap.index(shap_class_name)

            def _get_sv_2d_for_class(class_index: int):
                if isinstance(shap_values, list):
                    return shap_values[class_index]
                sv = _np.asarray(shap_values)
                if sv.ndim == 3:
                    return sv[:, :, class_index]
                if sv.ndim == 2:
                    return sv
                raise ValueError(f"Unexpected shap_values shape: {sv.shape}")

            def _get_shap_obj_for_class(class_index: int):
                if len(shap_obj.shape) == 2:
                    return -shap_obj if class_index == 0 else shap_obj
                return shap_obj[:, :, class_index]

            print('\nSHAP values summary plot for all classes')

            sv_for_summary = shap_values
            if not isinstance(sv_for_summary, list):
                sv_arr = _np.asarray(sv_for_summary)
                if sv_arr.ndim == 3:
                    sv_for_summary = [sv_arr[:, :, i] for i in range(sv_arr.shape[2])]
                else:
                    sv_for_summary = sv_arr

            shap.summary_plot(
                sv_for_summary, X_test,
                show=False, sort=True,
                class_names=CLASS_NAMES
            )

            ax = plt.gca()
            leg = ax.get_legend()

            if isinstance(sv_for_summary, list):
                n_classes = len(sv_for_summary)
                if n_classes > 1 and len(ax.patches) >= n_classes:
                    n_features = int(len(ax.patches) / n_classes)
                    class_inds = _np.argsort([-_np.abs(sv_for_summary[i]).mean() for i in range(n_classes)])

                    for i, c in enumerate(class_inds):
                        start = i * n_features
                        end = (i + 1) * n_features
                        for p in ax.patches[start:end]:
                            if c < len(CLASS_PALETTE):
                                p.set_facecolor(CLASS_PALETTE[c])

                    if leg is not None:
                        try:
                            handles = leg.legendHandles
                        except Exception:
                            try:
                                handles = leg.legend_handles
                            except Exception:
                                handles = None

                        if handles is not None:
                            for i, c in enumerate(class_inds):
                                if i < len(handles) and c < len(CLASS_PALETTE):
                                    try:
                                        handles[i].set_color(CLASS_PALETTE[c])
                                    except Exception:
                                        pass

            plt.show()

            w_shap_figheight = widgets.FloatSlider(value=2.0)

            def plotShapBar(shap_class):
                class_index = _get_class_index(shap_class)
                shap.plots.bar(_get_shap_obj_for_class(class_index), show=False)
                _, fig_height = plt.gcf().get_size_inches()
                w_shap_figheight.value = fig_height
                plt.show()

            def plotShapBeeswarm(shap_class, w_shap_figheight):
                class_index = _get_class_index(shap_class)
                shap.plots.beeswarm(_get_shap_obj_for_class(class_index), show=False)
                plt.gcf().set_figheight(w_shap_figheight)
                plt.show()

            def plotShapDependence(shap_class, main_index, interaction_index):
                class_index = _get_class_index(shap_class)
                sv2d = _get_sv_2d_for_class(class_index)
                shap.dependence_plot(
                    main_index,
                    sv2d,
                    X_test,
                    interaction_index=interaction_index,
                    show=False
                )
                plt.show()

            out_shap_bar = interactive_output(plotShapBar, {'shap_class': w_shap_class})
            out_shap_beeswarm = interactive_output(
                plotShapBeeswarm,
                {'shap_class': w_shap_class, 'w_shap_figheight': w_shap_figheight}
            )
            out_shap_dependence = interactive_output(
                plotShapDependence,
                {
                    'shap_class': w_shap_class,
                    'main_index': w_shap_feature0,
                    'interaction_index': w_shap_feature1
                }
            )

            print('\nSHAP values plot for selected class')
            shap_grid = widgets.GridspecLayout(1, 2, layout=widgets.Layout(justify_content='center'))
            shap_grid[0, 0] = out_shap_bar
            shap_grid[0, 1] = out_shap_beeswarm
            display(w_shap_class, shap_grid)

            print('\nDependence plots for selected features')
            display(
                widgets.HBox([w_shap_feature0, w_shap_feature1]),
                out_shap_dependence
            )

    # ---------------------
    # TAB: Zonal statistics
    # ---------------------
    with ctab.children[tab_idx['Zonal statistics']]:
        ctab.children[tab_idx['Zonal statistics']].clear_output()

        print('Select compute checkbox to calculate zonal areas on demand (cached after first run).')
        w_zonal_compute = widgets.Checkbox(
            value=False,
            description='Compute',
            disabled=False
        )

        out_zonal = widgets.Output()
        display(w_zonal_compute, out_zonal)

        def zonalComputeChanged(value):
            if value['new']:
                with out_zonal:
                    out_zonal.clear_output()

                    if temporal_interval == 'all-time':
                        print('Calculating zonal areas ...\n')
                        zonal_dict = zonalStatistics(image_classified)
                        _ = interact(
                            plotZonalAreas,
                            width=w_barWidth,
                            rot=w_rotation,
                            zonal_dict=fixed(zonal_dict)
                        )
                    else:
                        df_tmpZonal = getClassifiedZonalAreas(classify=not w_classified_tmpflag.value)

                        print('\n')
                        _ = interact(
                            plotTemporalZonal,
                            rot=w_rotation,
                            select=w_zonal_selectClasses,
                            chartType=w_zonal_chartType,
                            subplot=w_zonal_subplot,
                            df=fixed(df_tmpZonal)
                        )

        w_zonal_compute.observe(zonalComputeChanged, 'value')

    # ----------------------------
    # TAB: Spatio-temporal dynamics
    # ----------------------------
    if 'Spatio-temporal dynamics' in tab_idx:
        with ctab.children[tab_idx['Spatio-temporal dynamics']]:
            ctab.children[tab_idx['Spatio-temporal dynamics']].clear_output()
            desc_width1 = {'description_width': '160px'}

            tmp_dates = col_dates_list if 'col_dates_list' in globals() and col_dates_list is not None else []

            w_col_dates = widgets.SelectMultiple(
                options=tmp_dates,
                value=tuple(tmp_dates[:min(3, len(tmp_dates))]) if len(tmp_dates) > 0 else (),
                rows=6,
                description='Select dates',
                disabled=False,
                style=desc_width1
            )

            w_grid_column = widgets.BoundedIntText(
                value=2,
                min=1,
                max=6,
                step=1,
                description='Image columns',
                disabled=False,
                style=desc_width1
            )

            w_thumbnail_figsize = widgets.Text(
                value='10, 8',
                placeholder='In inches, separated by comma',
                description='Figure size',
                disabled=False,
                style=desc_width1
            )

            w_thumbnail_dimension = widgets.BoundedIntText(
                value=600,
                min=200,
                max=768,
                step=1,
                description='Thumbnail dimension',
                disabled=False,
                style=desc_width1
            )

            w_thumbnail = interactive(
                plotClassifiedGrid,
                {'manual': True, 'manual_name': 'Plot thumbnails in grid'},
                select_dates=w_col_dates,
                grid_column=w_grid_column,
                figsize=w_thumbnail_figsize,
                dim=w_thumbnail_dimension
            )

            w_btn_add2Map = widgets.Button(
                description='Add selected to map',
                button_style='',
                tooltip='Add classified image layers to map'
            )

            w_add2Map_output = widgets.Output()

            def btnAddMapLayers(obj):
                global listOfImages, colMosaic_tmp_classified, col_dates_list

                w_add2Map_output.clear_output()
                with w_add2Map_output:
                    try:
                        colMosaic_tmp_classified = getClassifiedCollections()

                        if listOfImages is None:
                            listOfImages = colMosaic_tmp_classified.toList(colMosaic_tmp_classified.size())

                        Map.layers = Map.layers[:5]
                        layer_lim = 5
                        resultVis1 = {'min': 0, 'max': len(CLASS_NAMES) - 1, 'palette': CLASS_PALETTE}

                        print(f'Add classified images up to {layer_lim} to map ...')

                        if col_dates_list is None:
                            print('No temporal dates available.')
                            return

                        selected_dates = list(w_col_dates.value)
                        if len(selected_dates) == 0:
                            print('Please select at least one date.')
                            return

                        col_index = [col_dates_list.index(d) for d in selected_dates if d in col_dates_list]

                        for s in col_index[:layer_lim]:
                            result = ee.Image(listOfImages.get(s))
                            Map.addLayer(result, resultVis1, f'Classified {col_dates_list[s]}')

                        print('Done. Use map layer managers to explore.')

                    except Exception as e:
                        print(f'Error adding layers: {e}')

            w_btn_add2Map.on_click(btnAddMapLayers)

            w_gif_dimension = widgets.BoundedIntText(
                value=600,
                min=200,
                max=768,
                step=1,
                description='Dimension',
                disabled=False,
                style=desc_width1
            )

            w_gif_fps = widgets.BoundedIntText(
                value=10,
                min=1,
                max=20,
                step=1,
                description='FramesPerSecond',
                disabled=False,
                style=desc_width1
            )

            w_gif_fname = widgets.Text(
                value='export_gif_classified',
                description='GIF file name',
                disabled=False,
                style=desc_width1
            )

            w_btn_export_gif = widgets.Button(
                description='Export all to GIF',
                button_style='',
                tooltip='Export image collection as GIF'
            )

            w_export_gif_output = widgets.Output()

            def btnExportGif(obj):
                global colMosaic_tmp_classified, col_dates_list

                w_export_gif_output.clear_output()
                with w_export_gif_output:
                    try:
                        colMosaic_tmp_classified = getClassifiedCollections()

                        print('Export the classified image collections ...')

                        framesPerSecond = w_gif_fps.value
                        video_args = {
                            'dimensions': w_gif_dimension.value,
                            'region': region,
                            'framesPerSecond': framesPerSecond,
                            'bands': ['vis-red', 'vis-green', 'vis-blue'],
                        }

                        out_gif = w_gif_fname.value + '.gif'
                        resultVis1 = {'min': 0, 'max': len(CLASS_NAMES) - 1, 'palette': CLASS_PALETTE}

                        tmp_rgb = colMosaic_tmp_classified.map(lambda img: ee.Image(img).visualize(**resultVis1))
                        tmp_rgb = ee.ImageCollection(tmp_rgb)

                        geemap.download_ee_video(tmp_rgb, video_args, out_gif)

                        print('Adding dates to GIF ...')
                        geemap.add_text_to_gif(
                            out_gif,
                            out_gif,
                            text_sequence=col_dates_list,
                            font_color='00ff00',
                            duration=1000 / framesPerSecond
                        )

                        print('Done. Please check', out_gif)
                        print()

                    except Exception as e:
                        print(f'Error exporting GIF: {e}')

            w_btn_export_gif.on_click(btnExportGif)

            w_export_gif = widgets.VBox([
                w_gif_dimension,
                w_gif_fps,
                w_gif_fname,
                w_btn_export_gif,
                w_export_gif_output
            ])

            section_export = widgets.VBox([
                widgets.HTML('<h3>Export classified image collection</h3>'),
                w_export_gif
            ])

            section_thumb = widgets.VBox([
                widgets.HTML('<h3>Display image thumbnails at selected dates</h3>'),
                w_thumbnail,
                w_btn_add2Map,
                w_add2Map_output
            ])

            display(widgets.VBox([section_export, section_thumb]))

    # ----------------------------
    # TAB: Olofsson Assessment
    # ----------------------------
    with ctab.children[tab_idx['Olofsson Assessment']]:
        ctab.children[tab_idx['Olofsson Assessment']].clear_output()

        print('Olofsson et al. 2014 Area Estimation Accuracy Assessment')
        print('=' * 60)
        print('Run on demand to avoid heavy computation during tab creation.')

        base_scale = int(scale) if 'scale' in globals() else 30

        w_olofsson_scale = widgets.BoundedIntText(
            value=base_scale,
            min=10,
            max=500,
            step=10,
            description='Scale (m)',
            style=desc_width
        )

        w_olofsson_tilescale = widgets.Dropdown(
            options=[4, 8, 16],
            value=16,
            description='Tile scale',
            style=desc_width
        )

        w_olofsson_force = widgets.Checkbox(
            value=False,
            description='Force recompute',
            style=desc_width
        )

        w_btn_run_olofsson = widgets.Button(
            description='Run Olofsson Assessment',
            button_style='danger',
            layout=widgets.Layout(width='260px')
        )

        out_olofsson = widgets.Output()

        display(widgets.VBox([
            widgets.HBox([w_olofsson_scale, w_olofsson_tilescale]),
            w_olofsson_force,
            w_btn_run_olofsson,
            out_olofsson
        ]))

        def _run_olofsson(_):
            out_olofsson.clear_output()

            with out_olofsson:
                try:
                    labels, y_true_eval, y_pred_eval, cm = getEvaluationArrays(
                        spatial_mode=spatial_mode,
                        holdout_mode=holdout_mode,
                        cv_fold_results=cv_fold_results if 'cv_fold_results' in globals() else None,
                        y_test=y_test,
                        y_pred=y_pred
                    )

                    params = {
                        'mode': 'spatial' if spatial_mode else 'holdout',
                        'scale': w_olofsson_scale.value,
                        'tile_scale': w_olofsson_tilescale.value,
                        'n_classes': len(CLASS_NAMES),
                    }

                    cached = analysis_cache.get('olofsson')
                    use_cache = (
                        cached is not None and
                        cached.get('params') == params and
                        not w_olofsson_force.value
                    )

                    if use_cache:
                        print('Using cached Olofsson results.')
                        Wi = cached['Wi']
                        total_area = cached['total_area']
                        pmatrix = cached['pmatrix']
                        olofsson_results = cached['results']
                    else:
                        print('Computing map class proportions ...')
                        Wi, total_area, area_dict = computeMapClassProportionsFast(
                            image_classified,
                            region,
                            scale=w_olofsson_scale.value,
                            tile_scale=w_olofsson_tilescale.value
                        )

                        print(f'Total mapped area: {total_area:.2f} km²')

                        pmatrix = buildAreaProportionedMatrix(cm, Wi, labels)
                        olofsson_results = computeOlofssonMetrics(cm, Wi, total_area, labels)

                        analysis_cache['olofsson'] = {
                            'params': params,
                            'Wi': Wi,
                            'total_area': total_area,
                            'pmatrix': pmatrix,
                            'results': olofsson_results
                        }

                    print('\nMap class proportions (Wi):')
                    for cls_idx, name in enumerate(CLASS_NAMES):
                        print(f'  {name}: {Wi.get(cls_idx, 0):.4f}')

                    print('\nArea-Proportioned Error Matrix:')
                    display(pmatrix.style.format('{:.4f}'))

                    print('\nAdjusted Metrics:')
                    display(olofsson_results.style.format({
                        'Map Area (km2)': '{:.3f}',
                        'Adjusted Area (km2)': '{:.3f}',
                        '95% CI (+/- km2)': '{:.3f}',
                        'Adjusted UA': '{:.3f}',
                        'Adjusted PA': '{:.3f}',
                    }))

                    oa_adjusted = sum(pmatrix.iloc[i, i] for i in range(len(CLASS_NAMES)))
                    print(f'\nArea-adjusted OA: {oa_adjusted:.3f}')

                    if spatial_mode:
                        print('Evaluation source: Spatial Block CV out-of-fold predictions.')
                        print('--- Spatial Block CV Summary ---')
                        # _cv_summary already computed above
                    else:
                        print('Evaluation source: Holdout test predictions.')

                except Exception as e:
                    print(f'Error computing Olofsson metrics: {e}')
                    import traceback
                    traceback.print_exc()

        w_btn_run_olofsson.on_click(_run_olofsson)

    # ----------------------------
    # TAB: Change Detection
    # ----------------------------
    if 'Change Detection' in tab_idx:
        with ctab.children[tab_idx['Change Detection']]:
            ctab.children[tab_idx['Change Detection']].clear_output()

            print('Post-Classification Change Detection')
            print('=' * 60)
            print('Run on demand to avoid executing change detection during classification.')

            base_scale = int(scale) if 'scale' in globals() else 30

            w_cd_scale = widgets.BoundedIntText(
                value=base_scale,
                min=10,
                max=500,
                step=10,
                description='Scale (m)',
                style=desc_width
            )

            w_cd_tilescale = widgets.Dropdown(
                options=[4, 8, 16],
                value=16,
                description='Tile scale',
                style=desc_width
            )

            w_cd_force = widgets.Checkbox(
                value=False,
                description='Force recompute',
                style=desc_width
            )

            w_btn_run_change = widgets.Button(
                description='Run Change Detection',
                button_style='danger',
                layout=widgets.Layout(width='240px')
            )

            out_change = widgets.Output()

            display(widgets.VBox([
                widgets.HBox([w_cd_scale, w_cd_tilescale]),
                w_cd_force,
                w_btn_run_change,
                out_change
            ]))

            def _run_change_detection(_):
                global transition_df, change_map, classified_period1, classified_period2

                out_change.clear_output()

                with out_change:
                    try:
                        if not (w_period1_start.value and w_period1_end.value and
                                w_period2_start.value and w_period2_end.value):
                            print('Please set all change detection date ranges.')
                            return

                        p1 = (
                            w_period1_start.value.strftime('%Y-%m-%d'),
                            w_period1_end.value.strftime('%Y-%m-%d')
                        )
                        p2 = (
                            w_period2_start.value.strftime('%Y-%m-%d'),
                            w_period2_end.value.strftime('%Y-%m-%d')
                        )

                        params = {
                            'period1': p1,
                            'period2': p2,
                            'scale': w_cd_scale.value,
                            'tile_scale': w_cd_tilescale.value,
                            'min_area': w_min_transition_area.value,
                            'n_classes': len(CLASS_NAMES),
                            'n_features': len(feature_names),
                        }

                        cached = analysis_cache.get('change_detection')
                        use_cache = (
                            cached is not None and
                            cached.get('params') == params and
                            not w_cd_force.value
                        )

                        if use_cache:
                            print('Using cached change detection results.')
                            classified_period1 = cached['classified_period1']
                            classified_period2 = cached['classified_period2']
                            transition_df = cached['transition_df']
                        else:
                            print(f'Processing Period 1: {p1[0]} -> {p1[1]}')
                            print(f'Processing Period 2: {p2[0]} -> {p2[1]}')

                            c1, c2 = classifyTwoPeriods(
                                p1, p2, region, feature_names,
                                training_data=LABEL_DATA,
                                label_col=LABEL,
                                retrain_per_period=True
                            )
                            if c1 is None or c2 is None:
                                print('Could not classify one or both periods.')
                                return

                            classified_period1 = c1
                            classified_period2 = c2

                            transition_df = computeTransitionMatrixFast(
                                c1, c2, CLASS_NAMES, region,
                                scale=w_cd_scale.value,
                                tile_scale=w_cd_tilescale.value
                            )

                            analysis_cache['change_detection'] = {
                                'params': params,
                                'classified_period1': c1,
                                'classified_period2': c2,
                                'transition_df': transition_df
                            }

                        _remove_map_layer_by_name('Change Map')
                        change_map = generateChangeMap(
                            classified_period1,
                            classified_period2,
                            len(CLASS_NAMES)
                        )

                        print('\nTransition Matrix (km²):')
                        display(transition_df.style.format('{:.2f}'))

                        # Validate transitions (flag small/noisy ones)
                        transition_df = validateTransitions(
                            transition_df, CLASS_NAMES,
                            min_area_km2=w_min_transition_area.value
                        )

                        print('\nRendering Sankey diagram...')
                        fig = renderSankeyDiagram(
                            transition_df,
                            CLASS_NAMES,
                            CLASS_PALETTE,
                            w_min_transition_area.value
                        )
                        fig.show()

                    except Exception as e:
                        print(f'Error in change detection tab: {e}')
                        import traceback
                        traceback.print_exc()

            w_btn_run_change.on_click(_run_change_detection)

    return ctab




# The APP

### CSS Style Manager

In [42]:
def create_app_styles():
    return widgets.HTML('''
    <style>
        /* ===== CSS Variables - Color Scheme ===== */
        /* Earth/satellite theme (blues, greens) */
        :root {
            --primary-color: #1a73e8;
            --primary-dark: #1557b0;
            --primary-light: #4285f4;
            --secondary-color: #34a853;
            --secondary-dark: #2d8e47;
            --accent-color: #ea4335;
            --accent-warning: #fbbc04;
            
            /*  Background colors for sections */
            --bg-light: #f8f9fa;
            --bg-card: #ffffff;
            --bg-sidebar: #f1f3f4;
            
            /* Sufficient contrast for accessibility */
            --text-primary: #202124;
            --text-secondary: #5f6368;
            --text-light: #80868b;
            --text-on-primary: #ffffff;
            
            /* Consistent accent colors */
            --border-color: #dadce0;
            --border-focus: #1a73e8;
            
            /* Shadows */
            --shadow-sm: 0 1px 2px rgba(0, 0, 0, 0.1);
            --shadow-md: 0 4px 6px rgba(0, 0, 0, 0.1);
            --shadow-lg: 0 10px 15px rgba(0, 0, 0, 0.1);
            --shadow-card: 0 2px 8px rgba(0, 0, 0, 0.08);
            
            /* Spacing */
            --spacing-xs: 4px;
            --spacing-sm: 8px;
            --spacing-md: 12px;
            --spacing-lg: 16px;
            --spacing-xl: 24px;
            
            /* Border radius */
            --radius-sm: 4px;
            --radius-md: 8px;
            --radius-lg: 12px;
            --radius-xl: 16px;
            
            /* Transitions */
            --transition-fast: 0.15s ease;
            --transition-normal: 0.25s ease;
        }
        
        /* ===== Card Styles ===== */
        .app-card {
            background: var(--bg-card);
            border: 1px solid var(--border-color);
            border-radius: var(--radius-md);
            margin-bottom: var(--spacing-sm);
            box-shadow: var(--shadow-card);
            overflow: hidden;
            transition: box-shadow var(--transition-normal);
        }
        
        .app-card:hover {
            box-shadow: var(--shadow-md);
        }
        
        .app-card-header {
            display: flex;
            align-items: center;
            padding: var(--spacing-md) var(--spacing-lg);
            background: linear-gradient(135deg, var(--bg-light) 0%, var(--bg-card) 100%);
            border-bottom: 1px solid var(--border-color);
            cursor: pointer;
            user-select: none;
        }
        
        .app-card-header:hover {
            background: linear-gradient(135deg, #e8eaed 0%, var(--bg-light) 100%);
        }
        
        .app-card-icon {
            font-size: 18px;
            margin-right: var(--spacing-sm);
        }
        
        .app-card-title {
            flex: 1;
            font-weight: 600;
            font-size: 14px;
            color: var(--text-primary);
        }
        
        .app-card-toggle {
            font-size: 12px;
            color: var(--text-secondary);
            transition: transform var(--transition-fast);
        }
        
        .app-card-toggle.collapsed {
            transform: rotate(-90deg);
        }
        
        .app-card-content {
            padding: var(--spacing-md);
        }
        
        .app-card-content.collapsed {
            display: none;
        }
        
        /* ===== Button Styles ===== */
        .app-btn-primary {
            background: linear-gradient(135deg, var(--primary-color) 0%, var(--secondary-color) 100%) !important;
            color: var(--text-on-primary) !important;
            border: none !important;
            border-radius: var(--radius-md) !important;
            padding: var(--spacing-md) var(--spacing-lg) !important;
            font-weight: 600 !important;
            font-size: 14px !important;
            cursor: pointer !important;
            transition: all var(--transition-normal) !important;
            box-shadow: var(--shadow-sm) !important;
        }
        
        .app-btn-primary:hover {
            background: linear-gradient(135deg, var(--primary-dark) 0%, var(--secondary-dark) 100%) !important;
            box-shadow: var(--shadow-md) !important;
            transform: translateY(-1px) !important;
        }
        
        .app-btn-primary:active {
            transform: translateY(0) !important;
            box-shadow: var(--shadow-sm) !important;
        }
        
        .app-btn-secondary {
            background: var(--bg-card) !important;
            color: var(--primary-color) !important;
            border: 2px solid var(--primary-color) !important;
            border-radius: var(--radius-md) !important;
            padding: var(--spacing-md) var(--spacing-lg) !important;
            font-weight: 600 !important;
            font-size: 14px !important;
            cursor: pointer !important;
            transition: all var(--transition-normal) !important;
        }
        
        .app-btn-secondary:hover {
            background: var(--primary-color) !important;
            color: var(--text-on-primary) !important;
        }
        
        /* ===== Widget Overrides ===== */
        /* Dropdown styling */
        .widget-dropdown select,
        .widget-select select {
            border: 1px solid var(--border-color) !important;
            border-radius: var(--radius-sm) !important;
            padding: var(--spacing-sm) var(--spacing-md) !important;
            font-size: 13px !important;
            transition: border-color var(--transition-fast), box-shadow var(--transition-fast) !important;
        }
        
        .widget-dropdown select:focus,
        .widget-select select:focus {
            border-color: var(--border-focus) !important;
            box-shadow: 0 0 0 2px rgba(26, 115, 232, 0.2) !important;
            outline: none !important;
        }
        
        /* Text input styling */
        .widget-text input,
        .widget-textarea textarea {
            border: 1px solid var(--border-color) !important;
            border-radius: var(--radius-sm) !important;
            padding: var(--spacing-sm) var(--spacing-md) !important;
            font-size: 13px !important;
            transition: border-color var(--transition-fast), box-shadow var(--transition-fast) !important;
        }
        
        .widget-text input:focus,
        .widget-textarea textarea:focus {
            border-color: var(--border-focus) !important;
            box-shadow: 0 0 0 2px rgba(26, 115, 232, 0.2) !important;
            outline: none !important;
        }
        
        /* Checkbox styling */
        .widget-checkbox input[type=\"checkbox\"] {
            width: 18px !important;
            height: 18px !important;
            accent-color: var(--primary-color) !important; 
        }
        
        /* Label styling */
        .widget-label-basic,
        .widget-dropdown > label,
        .widget-select > label,
        .widget-text > label {
            font-size: 13px !important;
            font-weight: 500 !important;
            color: var(--text-primary) !important;
        }
        
        /* DatePicker styling */
        .widget-datepicker input {
            border: 1px solid var(--border-color) !important;
            border-radius: var(--radius-sm) !important;
            padding: var(--spacing-sm) var(--spacing-md) !important;
            font-size: 13px !important;
        }
        
        .widget-datepicker input:focus {
            border-color: var(--border-focus) !important;
            box-shadow: 0 0 0 2px rgba(26, 115, 232, 0.2) !important;
            outline: none !important;
        }
        
        /* SelectMultiple styling */
        .widget-select-multiple select {
            border: 1px solid var(--border-color) !important;
            border-radius: var(--radius-sm) !important;
            font-size: 13px !important;
        }
        
        .widget-select-multiple select:focus {
            border-color: var(--border-focus) !important;
            box-shadow: 0 0 0 2px rgba(26, 115, 232, 0.2) !important;
            outline: none !important;
        }
        
        /* ===== Header Styles ===== */
        .app-header {
            background: linear-gradient(135deg, var(--primary-color) 0%, var(--secondary-color) 100%);
            padding: var(--spacing-lg) var(--spacing-xl);
            border-radius: var(--radius-lg);
            margin-bottom: var(--spacing-lg);
            box-shadow: 0 4px 15px rgba(26, 115, 232, 0.3);
        }
        
        .app-header-content {
            display: flex;
            align-items: center;
            justify-content: center;
        }
        
        .app-header-icon {
            font-size: 32px;
            margin-right: 15px;
        }
        
        .app-header-title {
            color: white;
            margin: 0;
            font-size: 24px;
            font-weight: 600;
            text-shadow: 0 2px 4px rgba(0, 0, 0, 0.2);
        }
        
        .app-header-subtitle {
            color: rgba(255, 255, 255, 0.9);
            margin: 5px 0 0 0;
            font-size: 14px;
        }
        
        /* ===== Map Panel Styles ===== */
        .app-map-container {
            border: 1px solid var(--border-color);
            border-radius: var(--radius-lg);
            overflow: hidden;
            box-shadow: var(--shadow-md);
        }
        
        .app-log-container {
            border: 1px solid var(--border-color);
            border-radius: var(--radius-md);
            margin-top: var(--spacing-lg);
            background: var(--bg-light);
            overflow: hidden;
        }
        
        .app-log-header {
            display: flex;
            align-items: center;
            padding: var(--spacing-sm) var(--spacing-md);
            background: var(--bg-card);
            border-bottom: 1px solid var(--border-color);
            font-weight: 600;
            font-size: 13px;
            color: var(--text-primary);
        }
        
        .app-log-header-icon {
            margin-right: var(--spacing-sm);
        }
        
        /* ===== Sidebar Styles ===== */
        .app-sidebar {
            background: var(--bg-sidebar);
            padding: var(--spacing-md);
            border-radius: var(--radius-md);
        }
    </style>
    ''')


In [43]:
def create_header():
    """
    Create styled application header with gradient background.
    
    Returns an HTML widget containing:
    - Gradient background from primary to secondary color
    - Satellite icon (🛰️)
    - Modern typography with title and subtitle
    - Appropriate padding and shadow for visual depth
    
    """
    return widgets.HTML('''
        <div class="app-header">
            <div class="app-header-content">
                <span class="app-header-icon">🛰️</span>
                <div>
                    <h1 class="app-header-title">
                        ỨNG DỤNG HỌC MÁY ĐÁNH GIÁ BIẾN ĐỘNG
                    </h1>
                    <p class="app-header-subtitle">
                        Sử dụng đất / Lớp phủ bề mặt đất • Google Earth Engine
                    </p>
                </div>
            </div>
        </div>
    ''')


In [44]:
def create_card(title, icon, children, collapsed=False):
    """
    Create a styled card component with collapsible functionality.
    
    Args:
        title (str): Card title text
        icon (str): Emoji icon for the card header
        children (list): List of widgets to include in card body
        collapsed (bool): Initial collapsed state (default: False)
    
    Returns:
        widgets.VBox: Styled card widget with header and content
    
    """
    toggle_icon = '▶' if collapsed else '▼'
    
    header_html = widgets.HTML(f'''
        <div style="
            display: flex;
            align-items: center;
            flex: 1;
        ">
            <span style="font-size: 18px; margin-right: 8px;">{icon}</span>
            <span style="
                flex: 1;
                font-weight: 600;
                font-size: 14px;
                color: #202124;
            ">{title}</span>
        </div>
    ''')
    
    toggle_btn = widgets.Button(
        description=toggle_icon,
        button_style='',
        layout=widgets.Layout(
            width='28px',
            height='28px',
            padding='0',
            margin='0',
            border='none',
            min_width='28px'
        ),
        tooltip='Click to collapse/expand'
    )
    toggle_btn.style.button_color = 'transparent'
    toggle_btn.style.font_weight = 'normal'
    
    # Create header container
    header = widgets.HBox(
        children=[header_html, toggle_btn],
        layout=widgets.Layout(
            display='flex',
            align_items='center',
            padding='12px 16px',
            border_bottom='1px solid #dadce0'
        )
    )
    header.add_class('app-card-header')
    
    # Create content container with children
    content = widgets.VBox(
        children=children,
        layout=widgets.Layout(
            padding='12px',
            display='none' if collapsed else 'flex',
            flex_flow='column',
            gap='8px'
        )
    )
    content.add_class('app-card-content')
    
    # Create the card container
    card = widgets.VBox(
        children=[header, content],
        layout=widgets.Layout(
            border='1px solid #dadce0',
            border_radius='8px',
            margin='8px 0',
            overflow='hidden',
            background='white'
        )
    )
    card.add_class('app-card')
    
    card._content = content
    card._toggle_btn = toggle_btn
    card._collapsed = collapsed
    
    def toggle_card(btn=None):
        """Toggle the card's collapsed state."""
        card._collapsed = not card._collapsed
        if card._collapsed:
            content.layout.display = 'none'
            toggle_btn.description = '▶'
        else:
            content.layout.display = 'flex'
            toggle_btn.description = '▼'
    
    toggle_btn.on_click(toggle_card)
    
    card.toggle = toggle_card
    
    return card


In [45]:
def create_primary_button(description, icon='', on_click_handler=None):
   
    btn_text = f'{icon} {description}'.strip() if icon else description
    
    btn = widgets.Button(
        description=btn_text,
        button_style='',
        layout=widgets.Layout(
            width='100%',
            height='44px',
            margin='8px 0',
            border_radius='8px'
        ),
        tooltip=description
    )
    
    # Apply primary button styling
    btn.add_class('app-btn-primary')
    
    # Attach click handler if provided
    if on_click_handler:
        btn.on_click(on_click_handler)
    
    return btn


def create_secondary_button(description, icon='', on_click_handler=None):
    """
    Create a styled secondary action button with outline style.
    
    Args:
        description (str): Button text
        icon (str): Optional emoji icon to prepend to description
        on_click_handler (callable): Optional click handler function
    
    Returns:
        widgets.Button: Styled secondary button widget
    
    """
    btn_text = f'{icon} {description}'.strip() if icon else description
    
    btn = widgets.Button(
        description=btn_text,
        button_style='',
        layout=widgets.Layout(
            width='100%',
            height='44px',
            margin='8px 0',
            border_radius='8px'
        ),
        tooltip=description
    )
    
    btn.add_class('app-btn-secondary')
    
    if on_click_handler:
        btn.on_click(on_click_handler)
    
    return btn


In [46]:
def create_sidebar():
    """Create the sidebar with organized card sections including new advanced modules."""

    card_region = create_card(
        title='Region of Interest',
        icon='',
        children=[w_regions],
        collapsed=False
    )

    card_dates = create_card(
        title='Time Period',
        icon='',
        children=[w_dates],
        collapsed=False
    )

    card_source = create_card(
        title='Data Source',
        icon='',
        children=[w_satellite, w_select_bands, w_source],
        collapsed=False
    )

    card_cloud = create_card(
        title='CloudScore+',
        icon='',
        children=[w_cloudscore_threshold],
        collapsed=True
    )

    # Only show Cloud Masking card when Sentinel-2 SR is selected
    is_s2_sr = w_satellite.value == 'Sentinel-2 SR'
    card_cloud.layout.display = 'block' if is_s2_sr else 'none'

    def _on_satellite_change_cloud(change):
        if change.new == 'Sentinel-2 SR':
            card_cloud.layout.display = 'block'
        else:
            card_cloud.layout.display = 'none'

    w_satellite.observe(_on_satellite_change_cloud, names='value')

    card_sar = create_card(
        title='SAR Fusion',
        icon='',
        children=[w_sar_fusion, w_speckle_radius],
        collapsed=True
    )

    card_phenology = create_card(
        title='Phenology',
        icon='',
        children=[w_harmonic_regression, w_harmonic_warning, w_harmonic_order, w_phenology_metrics, w_phenology_warning],
        collapsed=True
    )

    card_indices = create_card(
        title='Indices & Variables',
        icon='',
        children=[w_spectral_indices, w_variables],
        collapsed=False
    )

    card_validation = create_card(
        title='Validation',
        icon='',
        children=[
            w_spatial_block_cv, w_block_size_km, w_n_folds, w_block_warning,
            w_split_ratio
        ],
        collapsed=True
    )

    card_classifier = create_card(
        title='Classifier',
        icon='',
        children=[w_classifier],
        collapsed=False
    )

    card_sample = create_card(
        title='Training Samples',
        icon='',
        children=[w_sample],
        collapsed=False
    )

    card_change = create_card(
        title='Change Detection',
        icon='',
        children=[w_change_detection, w_change_detection_detail],
        collapsed=True
    )

    card_execute = create_card(
        title='Execute',
        icon='',
        children=[w_btn_classify],
        collapsed=False
    )

    sidebar = widgets.VBox(
        children=[
            card_region,
            card_dates,
            card_source,
            card_cloud,
            card_sar,
            card_indices,
            card_phenology,
            card_validation,
            card_classifier,
            card_sample,
            card_change,
            card_execute
        ],
        layout=widgets.Layout(
            width='320px',
            padding='10px',
            overflow_y='auto',
            max_height='85vh',
            background='#f8f9fa'
        )
    )

    return sidebar

In [47]:
def create_map_panel(map_widget, log_output):
    map_container = widgets.VBox(
        children=[map_widget],
        layout=widgets.Layout(
            border='1px solid #dadce0',
            border_radius='12px',
            overflow='hidden',
            box_shadow='0 2px 8px rgba(0,0,0,0.1)',
            height='800px'
        )
    )
    
    log_header = widgets.HTML('''
        <div style="
            background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
            padding: 10px 15px;
            border-bottom: 1px solid #dadce0;
            font-weight: 600;
            color: #202124;
            display: flex;
            align-items: center;
            gap: 8px;
        ">
            <span style="font-size: 16px;">📋</span>
            <span>Workflow Log</span>
        </div>
    ''')
    
    log_output.layout = widgets.Layout(
        height='180px',
        overflow='auto',
        padding='10px',
        background='#ffffff'
    )
    
    log_container = widgets.VBox(
        children=[log_header, log_output],
        layout=widgets.Layout(
            border='1px solid #dadce0',
            border_radius='8px',
            margin_top='15px',
            background='#f8f9fa',
            overflow='hidden'
        )
    )
    
    map_panel = widgets.VBox(
        children=[map_container, log_container],
        layout=widgets.Layout(
            flex='1',
            padding='0 0 0 20px',
            width='72%'
        )
    )
    
    return map_panel


In [48]:
def runApp():
    
    css_styles = create_app_styles()
    
    header = create_header()
    
    
    left_sidebar = create_sidebar()
    
   
    map_panel = create_map_panel(Map, w_btn_workflow_log)
    
    main_content = widgets.HBox(
        children=[left_sidebar, map_panel],
        layout=widgets.Layout(
            display='flex',
            width='100%'
        )
    )
    
    
    app = widgets.VBox(
        children=[
            css_styles,  
            header,      
            main_content 
        ],
        layout=widgets.Layout(
            padding='10px',
            background='#f8f9fa'
        )
    )
    
    display(app)

runApp()

## Feature importance tab

In [49]:
ctab = createTabs() #chạy classify ở display app rồi mới chạy cell này


Classify the image first.
